# Notebook 11 - Learning Interface Validation

## Purpose

Notebook 11 performs the final learning-interface validation for the initial HydroLab-3D notebook-development sequence.

The central question is deliberately narrow:

> **Can a learning algorithm interact with the completed HydroLab-3D production `Simulator`, through an external temporary environment wrapper, and demonstrate measurable improvement without requiring the simulator core to be redesigned?**

This notebook is fundamentally different from Notebooks 00 through 10.

The earlier notebooks progressively developed, validated, and where appropriate promoted reusable simulator functionality into the production package. Notebook 11 does **not** begin another round of simulator construction.

Instead, it treats the completed production simulator as the plant under test:

```text
learning algorithm
        ↓
temporary learning interface
        ↓
hydrolab3d.core.Simulator
        ↓
existing production dynamics + currents + sensing
        ↓
updated simulator state
        ↺
```

The learning code exists outside the simulator.

HydroLab-3D itself remains independent of:

```text
Gymnasium
Stable-Baselines3
PPO
PyTorch
reward functions
episode definitions
learning-task conventions
```

The objective is compatibility validation rather than the creation of a permanent reinforcement-learning subsystem.

---

## Why This Notebook Exists

A simulator should not be described as suitable for learning-based research merely because it has a function that resembles:

```text
step(action)
```

A usable learning environment requires a stronger interaction contract.

At minimum, an external learning system must be able to perform the following loop repeatedly and predictably:

```text
reset environment
      ↓
construct observation
      ↓
select action
      ↓
map learning action to simulator action
      ↓
advance production Simulator
      ↓
calculate reward
      ↓
test termination / truncation
      ↓
construct next observation
      ↺
```

This requires more than physical correctness.

The complete loop must also demonstrate:

```text
consistent dimensions
finite numerical values
bounded actions
well-defined reset behavior
repeatable stochastic behavior
stable long interaction sequences
valid episode boundaries
fair evaluation
measurable learning
```

Notebook 11 therefore acts as an **external integration test between HydroLab-3D and a learning algorithm**.

It validates that the simulator architecture built during the previous notebooks is sufficiently clean that reinforcement learning can be added from outside rather than embedded into the core.

That architectural distinction is important.

The desired result is:

$$
\boxed{
\text{learning compatibility}
\neq
\text{learning-framework dependency}
}
$$

---

## Relationship to Notebooks 00-10

Notebook 11 inherits the entire validated HydroLab-3D engineering lineage.

```text
Notebook 00
Basic 3D Kinematics
        ↓
Notebook 01
Orientation and Reference Frames
        ↓
Notebook 02
Basic Dynamics
        ↓
Notebook 03
Hydrodynamic Drag
        ↓
Notebook 04
Buoyancy and Gravity
        ↓
Notebook 05
Thruster Model
        ↓
Notebook 06
Ocean Currents
        ↓
Notebook 07
Integrated 6-DOF UUV Dynamics
        ↓
Notebook 08
Sensor Models
        ↓
Notebook 09
Closed-Loop Control
        ↓
Notebook 10
Production Simulator Kernel
        ↓
Notebook 11
Learning Interface Validation
```

The previous notebooks have already established the mathematical and software foundations required here.

Notebook 11 therefore does **not** reopen the derivation of:

```text
3D kinematics
rotation matrices
rigid-body acceleration
hydrodynamic drag
hydrostatic restoring forces
thruster physics
ocean-current interaction
6-DOF UUV dynamics
sensor equations
PID control
waypoint guidance
simulator orchestration
```

unless execution in this notebook reveals a genuine integration defect.

The immediate predecessor, Notebook 10, completed the production simulator boundary and finished with:

```text
Dedicated Simulator tests : 15 passed
Complete repository suite : 167 passed
Failures                  : 0
```

That is the inherited regression baseline for Notebook 11.

Notebook 11 begins by trusting those validated results while independently verifying that the resulting public interface can support learning.

---

## Inherited Production Simulator

The principal interface inherited from Notebook 10 is:

```python
from hydrolab3d.core import Simulator
```

The production kernel is constructed from:

```text
initial pose
initial body velocity
fixed timestep
6-DOF dynamics configuration
sensor configuration
sensor RNG seed
```

and owns:

```text
simulation truth
deterministic model time
vehicle/environment configuration
sensor configuration
sensor random-number state
```

The public lifecycle is:

```python
simulator = Simulator(...)

state = simulator.reset()

result = simulator.step(action)
```

The production generalized action is a six-component body-frame wrench:

$$
\tau
=
\begin{bmatrix}
F_x &
F_y &
F_z &
M_x &
M_y &
M_z
\end{bmatrix}^{T}.
$$

Therefore:

```text
Simulator action
=
[Fx, Fy, Fz, Mx, My, Mz]
```

with translational components expressed as forces and rotational components expressed as moments.

Notebook 11 will **not modify this contract** merely to make it resemble a conventional reinforcement-learning action space.

Any reduced action representation exposed to a learning algorithm will instead be transformed explicitly by the temporary notebook-local wrapper.

---

## Simulator Truth and Sensor Observation

Notebook 10 established a strict separation between physical ground truth and simulated sensor measurements.

The production step result has the conceptual structure:

```text
result
│
├── truth
│   ├── time
│   ├── pose
│   └── body_velocity
│
└── observation
    ├── depth
    ├── heading
    ├── dvl
    ├── gyroscope
    └── accelerometer
```

The authoritative truth state uses:

$$
\eta
=
[x,y,z,\phi,\theta,\psi]^T
$$

for pose and:

$$
\nu
=
[u,v,w,p,q,r]^T
$$

for body-frame velocity.

The sensor layer provides imperfect measurements derived from that truth.

The architectural invariant remains:

$$
\boxed{
\text{truth}
\neq
\text{observation}
}
$$

Notebook 11 will preserve this distinction explicitly.

---

## Learning-Validation Philosophy

The purpose of the learning experiment is **not** to demonstrate sophisticated reinforcement-learning performance.

The test is successful if a small learning algorithm can:

1. interact repeatedly with the real HydroLab-3D production simulator;
2. receive a well-defined numerical observation;
3. produce a bounded learning action;
4. have that action mapped explicitly into the production six-axis wrench;
5. accumulate rewards through a coherent task;
6. complete valid episodes;
7. survive random-action stress testing;
8. train for a modest number of interactions;
9. produce behavior that measurably improves over an appropriate baseline.

The desired evidence chain is:

```text
stable simulator interaction
        ↓
stable random-policy execution
        ↓
baseline performance measurement
        ↓
short learning run
        ↓
independent evaluation
        ↓
measurable improvement
```

A visually attractive trajectory alone will not constitute evidence of learning.

Likewise, a rising training reward alone will not be sufficient.

The final conclusion must be based on evaluation metrics computed after training.

---

## Notebook-Local Wrapper Architecture

The temporary learning environment will remain entirely inside this notebook.

Conceptually:

```text
                 PPO / random policy
                        │
                        ▼
              normalized learning action
                        │
                        ▼
             Notebook-local environment
               │        │        │
               │        │        └── reward / episode logic
               │        │
               │        └── observation construction
               │
               └── action mapping
                        │
                        ▼
            six-axis generalized wrench
                        │
                        ▼
             hydrolab3d.core.Simulator
                        │
                        ▼
              production 6-DOF physics
                        │
                        ▼
               truth + sensor output
                        │
                        └───────────────↺
```

The wrapper may implement the standard Gymnasium interaction convention:

```text
reset()
step(action)
observation_space
action_space
terminated
truncated
info
```

but none of those abstractions will be added to:

```text
src/hydrolab3d/
```

during this notebook.

---

## Learning Task

The initial task will remain intentionally simple:

> **Move one UUV from a fixed initial state toward a fixed three-dimensional waypoint.**

This is not intended to become a general navigation benchmark.

A fixed point-to-point task is useful because it isolates the interface question from unrelated problems such as:

```text
mapping
SLAM
terrain reasoning
obstacle avoidance
mission planning
communication
multi-agent coordination
perception
localization
```

None of those capabilities are required to establish whether the simulator can support learning.

The task can therefore be represented conceptually as:

```text
initial UUV state
        ↓
3D target waypoint
        ↓
policy chooses translational command
        ↓
temporary action mapper
        ↓
6-DOF HydroLab wrench
        ↓
vehicle moves
        ↓
distance to waypoint changes
        ↓
reward
```

---

## Observation Design

For this first learning smoke test, the policy will deliberately use **simulator ground truth** rather than pretending that HydroLab-3D already contains a complete underwater localization system.

This choice will be stated explicitly throughout the notebook.

Using ground truth here does **not** mean that HydroLab-3D's sensor/truth separation has been abandoned.

Instead, it isolates the question being tested:

> Can a learning policy control the finished physical simulator through an external interface?

A purely sensor-derived waypoint-navigation observation would require a localization or state-estimation layer that does not currently exist as part of the validated production simulator.

Inventing such a subsystem merely for Notebook 11 would mix two different research questions.

The initial policy observation will therefore be task-relative and truth-derived.

A suitable representation is:

$$
o_t
=
\begin{bmatrix}
\Delta x &
\Delta y &
\Delta z &
\phi &
\theta &
\psi &
u &
v &
w &
p &
q &
r
\end{bmatrix}^{T}
$$

where:

$$
\begin{bmatrix}
\Delta x\\
\Delta y\\
\Delta z
\end{bmatrix}
=
\mathbf{p}_{goal}
-
\mathbf{p}_{vehicle}.
$$

Thus the observation contains:

```text
world-frame relative waypoint vector : 3 values
vehicle orientation                  : 3 values
body-frame linear velocity           : 3 values
body-frame angular velocity          : 3 values
                                      ----------
total                                : 12 values
```

The frame distinction will remain explicit:

```text
goal displacement : world frame
orientation       : roll / pitch / yaw
velocity          : body frame
```

This is intentionally more physically complete than feeding only position and velocity while silently assuming that orientation can never matter.

The wrapper may normalize or scale these quantities for learning-library numerical conditioning, but any such transformation will be documented explicitly.

Raw simulator state will remain unchanged.

---

## Why Ground Truth Is Acceptable Here

The long-term HydroLab-3D architecture should support algorithms that operate on imperfect measurements rather than privileged world knowledge.

Notebook 11 is not claiming otherwise.

The distinction is:

```text
Notebook 11 question:
Can a learning algorithm operate the production simulator?
```

versus:

```text
future question:
Can an underwater learning policy navigate from realistic onboard sensing
and uncertain localization?
```

The second question requires capabilities beyond this notebook.

Therefore:

$$
\boxed{
\text{ground-truth learning smoke test}
\neq
\text{sensor-based autonomy claim}
}
$$

The notebook will describe the learned policy accurately as a **truth-state point-navigation smoke test**.

A later research project can replace the observation constructor with:

```text
sensor measurements
state estimator
learned perception
local belief state
```

without changing the underlying `Simulator.step()` interface.

---

## Learning Action Design

The learning algorithm does not need to manipulate the full six-dimensional wrench during this first test.

A deliberately reduced action will be used:

$$
a_t
=
\begin{bmatrix}
a_x &
a_y &
a_z
\end{bmatrix}^{T},
$$

with normalized bounds:

$$
a_i \in [-1,1].
$$

These values will represent commanded translational effort rather than direct position or velocity commands.

The wrapper will map them into physical body-frame forces:

$$
F_x = F_{x,\max}a_x,
$$

$$
F_y = F_{y,\max}a_y,
$$

$$
F_z = F_{z,\max}a_z.
$$

The production action passed to HydroLab-3D will then be:

$$
\tau_t
=
\begin{bmatrix}
F_x &
F_y &
F_z &
0 &
0 &
0
\end{bmatrix}^{T}.
$$

Conceptually:

```text
RL action
[a_x, a_y, a_z]
        ↓
scale by explicit force limits
        ↓
[Fx, Fy, Fz]
        ↓
append zero moments
        ↓
[Fx, Fy, Fz, 0, 0, 0]
        ↓
Simulator.step(...)
```

The exact force scales will be selected only after auditing the actual production configuration inherited from Notebook 10.

They will not be guessed from the README.

This preserves the simulator's six-axis action contract while keeping the learning problem intentionally small.

It also means the policy cannot directly overwrite:

```text
position
velocity
orientation
acceleration
```

The policy must act through the validated vehicle dynamics.

---

## Reward Concept

The reward should encourage actual progress toward the waypoint while remaining simple enough to understand.

Let:

$$
d_t
=
\left\|
\mathbf{p}_{goal}
-
\mathbf{p}_t
\right\|_2
$$

be the vehicle's Euclidean distance from the target.

Rather than relying only on the absolute penalty:

$$
-d_t,
$$

the notebook will prefer a progress-oriented reward structure based on:

$$
d_{t-1}-d_t.
$$

Conceptually:

$$
r_t
=
w_p(d_{t-1}-d_t)
-
w_a\|a_t\|_2^2
-
w_t
+
r_{success}.
$$

The terms have clear meanings:

```text
progress term
    reward movement toward the target

action penalty
    discourage unnecessarily large continuous control effort

small step penalty
    mildly prefer reaching the target sooner

success bonus
    clearly reward entry into the target region
```

This formulation avoids granting large cumulative reward merely for existing near the target while making directional progress visible to the learner.

The reward is part of the temporary task definition.

It is **not** part of HydroLab-3D physics and will not enter the production package.

The final numerical weights will be kept deliberately modest and documented when the task contract is instantiated.

No reward-engineering sweep is planned.

---

## Success Region

The target will be represented by a finite three-dimensional success region rather than requiring the vehicle to reach one floating-point coordinate exactly.

Success occurs when:

$$
d_t
\le
r_{goal},
$$

where:

$$
r_{goal}
$$

is a small task-defined radius around the waypoint.

This provides a physically meaningful completion condition:

```text
vehicle enters goal sphere
        ↓
terminated = True
```

The exact radius will be selected relative to the task scale after the inherited dynamics and timestep are inspected.

---

## Episode Termination and Truncation

Gymnasium distinguishes between:

```text
termination
```

and:

```text
truncation
```

and Notebook 11 will preserve that distinction.

### Termination

A normal successful episode terminates when:

$$
d_t \le r_{goal}.
$$

If an unrecoverable invalid physical state is encountered, such as:

```text
NaN
Inf
```

the stress-test logic will treat it as an environment failure rather than quietly continuing training.

### Truncation

An episode will be truncated after a fixed maximum number of simulator steps:

$$
N \ge N_{max}.
$$

Therefore:

```text
goal reached
    → terminated

time horizon exhausted
    → truncated
```

If a notebook-local task boundary is introduced to prevent an obviously runaway policy from travelling arbitrarily far from the learning region, that boundary will be explicitly described as an **RL task boundary**, not as seabed, collision physics, or a production HydroLab world boundary.

Notebook 11 will not invent environment geometry that the simulator does not model.

---

## Reset Semantics

Environment reset behavior must be unambiguous.

For the minimal smoke test, reset will restore a fixed benchmark problem:

```text
same physical initial state
same target waypoint
simulation time = 0
fresh episode bookkeeping
```

At the simulator level:

```text
Simulator.reset()
```

already restores:

```text
initial pose
initial body velocity
simulation time
sensor RNG sequence
```

according to the Notebook 10 contract.

The environment wrapper will additionally restore its own task variables:

```text
previous goal distance
episode step count
episode return
termination state
```

The initial learning observation will then be constructed from the reset truth state.

The wrapper must not depend on an implicit previous episode.

---

## Randomness and Reproducibility

Learning experiments contain several distinct sources of stochasticity.

Notebook 11 will keep them conceptually separate.

Where applicable, explicit seeds will be assigned for:

```text
Simulator sensor RNG
environment RNG
Gymnasium action space
learning algorithm
evaluation
```

These are different stochastic systems and should not be collapsed into one vague global seed.

Conceptually:

```text
SIMULATOR_SENSOR_SEED
ENVIRONMENT_SEED
ACTION_SPACE_SEED
LEARNING_SEED
EVALUATION_SEED
```

The exact values will be recorded in the notebook.

Because the initial policy observation is truth-derived, sensor noise will not directly affect the policy input.

Nevertheless, the simulator's sensor RNG behavior will remain intact and untouched.

Training and evaluation will not intentionally share uncontrolled stochastic state.

---

## Dependency Boundary

Notebook 11 may use:

```text
gymnasium
stable-baselines3
torch
```

only as external validation tools.

Before importing them as required dependencies, the notebook will inspect the actual active environment.

The sequence will be:

```text
inspect project/environment
        ↓
check package availability
        ↓
identify versions
        ↓
only then construct learning wrapper
```

A missing ML dependency is an environment/dependency issue.

It is not evidence of a HydroLab-3D simulator failure.

Likewise, Notebook 11 will not silently modify permanent project dependencies merely because an optional smoke-test library is absent.

---

## Random-Policy Stress Test

Before PPO is allowed to train, the temporary environment must survive a random-action stress test.

This stage is mandatory.

The random policy deliberately explores the action space without intelligence:

```text
reset
  ↓
sample bounded random action
  ↓
environment step
  ↓
validate returned values
  ↓
repeat
```

The test will check, where applicable:

```text
observation shape
observation dtype
observation-space containment
action shape
action bounds
reward finiteness
truth-state finiteness
observation finiteness
simulation-time progression
step-count progression
termination behavior
truncation behavior
repeated resets
seed behavior
unexpected exceptions
NaN
Inf
```

It will also verify that the reduced action is mapped into the production wrench exactly as documented.

No PPO training will begin until this contract is stable.

This ordering is intentional:

$$
\boxed{
\text{debug environment first}
\rightarrow
\text{train policy second}
}
$$

Otherwise, an interface bug can easily masquerade as a reinforcement-learning problem.

---

## Learning Smoke Test

Once random interaction is stable, Notebook 11 will perform a deliberately small reinforcement-learning experiment.

The likely learning algorithm is:

```text
Proximal Policy Optimization (PPO)
```

through Stable-Baselines3.

PPO is appropriate here because the planned learning action space is continuous:

$$
a_t\in[-1,1]^3.
$$

The purpose is **not** to determine whether PPO is the best controller for this vehicle.

The purpose is not to compare algorithms.

The notebook will therefore avoid:

```text
PPO vs SAC
PPO vs TD3
large hyperparameter sweeps
architecture search
millions of unnecessary training steps
GPU-heavy optimization
reward-function competitions
```

A small CPU-friendly run is enough if it establishes measurable learning.

Training effort may be increased modestly only if the initial run is insufficient to answer the validation question.

---

## Baseline Philosophy

A trained policy needs a meaningful reference.

The primary baseline will be a random bounded policy operating through the **same environment and action space**.

Therefore both policies experience:

```text
same simulator
same initial state
same target
same timestep
same action limits
same episode horizon
same reward definition
same success definition
```

The only meaningful difference is:

```text
random action selection
```

versus:

```text
learned action selection
```

This avoids comparing two different tasks while calling the difference "learning."

If useful, the notebook may also inspect the untrained PPO policy before optimization, but random control remains the clean external baseline.

---

## Evaluation Metrics

Training reward alone is insufficient evidence.

Notebook 11 will evaluate behavior using task-space quantities that have direct physical meaning.

The primary candidate metrics are:

### Final Goal Distance

$$
d_{final}
=
\left\|
\mathbf{p}_{goal}
-
\mathbf{p}_{final}
\right\|_2.
$$

### Minimum Goal Distance

$$
d_{min}
=
\min_t
\left\|
\mathbf{p}_{goal}
-
\mathbf{p}_t
\right\|_2.
$$

### Success Rate

$$
S
=
\frac{
N_{success}
}{
N_{episodes}
}.
$$

### Episode Return

$$
G
=
\sum_{t=0}^{T} r_t.
$$

### Steps to Success

For successful episodes:

$$
N_{success\ steps}
=
\text{number of environment transitions before entering the goal region}.
$$

Only metrics relevant to the final environment contract will be retained.

The final comparison will emphasize physical task performance, particularly:

```text
goal distance
success rate
```

rather than presenting a neural-network loss as if it were navigation performance.

---

## Fair Evaluation

The trained policy will be evaluated separately from training.

Evaluation episodes will not be selected because they produce attractive trajectories.

Where stochastic policies or random baselines are involved, multiple episodes will be used.

Because the initial benchmark uses a fixed starting condition and waypoint, every policy will face the same physical task.

A representative comparison is:

```text
random policy
    × N evaluation episodes

trained policy
    × N evaluation episodes
```

followed by comparison of:

```text
mean final distance
median final distance
minimum distance
success count
mean episode return
```

where appropriate.

If PPO evaluation is performed deterministically, that fact will be stated explicitly.

If the fixed deterministic benchmark produces identical learned-policy trajectories across repetitions, the notebook will not pretend those repetitions are statistically independent samples.

---

## What Counts as Measurable Learning

Notebook 11 does not require a perfect policy.

The policy does not need to demonstrate:

```text
optimal control
minimum-energy navigation
high-precision station keeping
robust current rejection
generalization to arbitrary goals
sensor-based navigation
real-world transfer
```

The smoke test needs considerably less.

A successful learning result should show consistent evidence such as:

```text
trained policy finishes closer to the target than random control
```

and preferably:

```text
trained policy enters the goal region more frequently
```

or:

```text
trained policy achieves materially better episode return
```

under the same evaluation contract.

The result must be numerical rather than purely visual.

Therefore the required logical conclusion is:

$$
\boxed{
\text{random behavior}
\rightarrow
\text{training}
\rightarrow
\text{measurably improved task behavior}
}
$$

---

## Production-Code Boundary

Notebook 11 begins with the strong presumption:

```text
NO new production ML module
```

The following will remain notebook-local:

```text
Gymnasium environment class
observation constructor
observation normalization
action mapping
reward function
goal definition
episode bookkeeping
termination logic
truncation logic
PPO configuration
training code
evaluation code
learning-specific plots
```

No directory such as:

```text
src/hydrolab3d/rl/
src/hydrolab3d/learning/
src/hydrolab3d/envs/
```

will be created merely because the smoke test works.

Likewise, the following existing empty scaffolds will remain untouched unless an independent production requirement appears:

```text
core/clock.py
core/entity.py
core/state.py
```

The core Simulator should remain usable by someone who has never installed Gymnasium, Stable-Baselines3, or PyTorch.

---

## Genuine Core-Defect Exception

Notebook 11 is allowed to modify production code only if execution uncovers a real simulator defect that would also affect non-learning users.

If such a problem appears, the notebook must first identify the failing layer.

Possible failure classes include:

```text
temporary environment bug
action-mapping bug
reward/task-design issue
learning-library issue
package/import issue
production Simulator defect
underlying physics limitation
```

A failure during PPO training is not automatically a Simulator bug.

Production code will be changed only if the defect can be demonstrated independently of the learning algorithm.

If that occurs, the process will be:

```text
prove defect
    ↓
identify smallest production fix
    ↓
add appropriate permanent regression test
    ↓
rerun existing tests
    ↓
revalidate learning interface
```

Otherwise:

```text
src/hydrolab3d/
```

remains unchanged throughout Notebook 11.

---

## Artifact Policy

Notebook 11 owns exactly one persistent result namespace:

```text
results/notebooks/11_learning_interface_validation/
```

No previous notebook's artifacts will be modified.

Subdirectories will be created only when justified:

```text
data/
plots/
models/
logs/
```

Notebook 11 will generate only evidence that helps answer the learning-compatibility question.

A compact final artifact set will likely contain:

```text
evaluation metrics
one representative trajectory comparison
training-progress evidence if useful
optional saved smoke-test policy
```

A model file will be preserved only if it provides useful reproducibility value.

Decorative figures will not be generated merely to populate directories.

---

## Expected Validation Flow

The exact number of implementation steps is intentionally not fixed in advance.

The likely evidence chain is:

```text
inspect inherited production state
        ↓
audit optional learning dependencies
        ↓
define explicit task contract
        ↓
construct notebook-local environment
        ↓
validate reset
        ↓
validate one-step interaction
        ↓
stress with random actions
        ↓
validate reproducibility
        ↓
measure random baseline
        ↓
run small PPO smoke test
        ↓
evaluate learned policy
        ↓
compare learned vs random behavior
        ↓
write compact Notebook 11 artifacts
        ↓
rerun complete HydroLab-3D pytest suite
        ↓
final conclusion
```

This sequence may change if actual execution provides evidence that a step should be:

```text
added
removed
split
combined
```

The notebook will remain evidence-driven rather than forcing the experiment into a predetermined number of cells.

---

## Reproducibility Standard

A learning notebook introduces enough stochasticity that reproducibility must be treated as an engineering property rather than an afterthought.

The notebook will record:

```text
Python/library environment where relevant
Simulator timestep
vehicle configuration
initial state
goal state
force limits
reward parameters
episode horizon
success radius
all relevant seeds
PPO configuration
training interaction count
evaluation protocol
```

Repeated environment resets must have defined behavior.

Random and trained-policy evaluation must not accidentally inherit hidden state from training.

Persistent data should contain enough information to reconstruct the reported comparison without relying only on the plotted figure.

---

## Regression Requirement

Notebook 11 inherits:

$$\boxed{167\ \text{passed},\quad 0\ \text{failed}}$$

from Notebook 10.

Near the end of the notebook, the complete existing HydroLab-3D repository test suite will be rerun from the correctly configured `src/` environment.

Notebook 11 does not need to create permanent tests for notebook-only Gymnasium or PPO code.

Its own interface validation may live primarily in:

```text
assertions
stress-test summaries
evaluation metrics
persistent artifacts
```

Permanent pytest tests will be added only if permanent production behavior changes.

The notebook will not claim completion if the inherited production regression suite fails.

---

## Scope Boundaries

Notebook 11 will **not** become:

```text
a new simulator
a second UUV dynamics implementation
a production Gymnasium package
a general-purpose RL benchmark
a multi-agent RL environment
a control-algorithm competition
a reward-engineering study
a hyperparameter-search project
a neural-network architecture study
a localization project
a SLAM project
a sonar project
an obstacle-avoidance project
a terrain-contact project
a sim-to-real RL experiment
```

It also will not silently introduce physical capabilities that HydroLab-3D does not yet contain.

In particular, a numerical task boundary will not be described as collision geometry.

A target waypoint will not be described as sensed by sonar.

Ground-truth position will not be described as an onboard localization estimate.

The meaning of every learning signal will remain explicit.

---

## Known Limitations

Even if Notebook 11 succeeds, the demonstrated result will remain intentionally narrow.

The experiment will validate:

```text
single vehicle
single fixed point-navigation task
truth-derived policy observation
reduced translational learning action
existing lightweight 6-DOF physics
short PPO training
notebook-local environment
```

It will **not** establish:

```text
general RL robustness
arbitrary-waypoint generalization
sensor-only navigation
uncertain localization
obstacle avoidance
terrain interaction
multi-UUV learning
communication-aware RL
real-world transfer
optimal vehicle control
```

The inherited simulator also continues to use Euler-angle orientation, including the existing ZYX singularity near:

$$
\theta = \pm 90^\circ.
$$

Notebook 11 will not replace that representation.

These limitations do not invalidate the smoke test.

They define exactly what the result does and does not prove.

---

## Success Criteria

Notebook 11 will be considered complete only if all of the following are demonstrated:

### Interface Integrity

```text
environment resets correctly
environment steps correctly
observations have the declared shape
actions remain within declared bounds
learning actions map correctly to Simulator wrench
rewards remain finite
episode state remains coherent
```

### Numerical Stability

```text
random-action rollouts do not produce unexplained exceptions
expected simulation states remain finite
repeated resets remain valid
termination and truncation behave correctly
```

### Reproducibility

```text
relevant seeds are explicit
reset semantics are demonstrated
evaluation procedure is reproducible
```

### Learning Compatibility

```text
a learning algorithm successfully consumes observations
a learning algorithm successfully produces actions
training executes through the real production Simulator
```

### Measurable Improvement

```text
trained behavior
        >
appropriate random baseline
```

according to declared task metrics rather than visual impression alone.

### Production Integrity

```text
existing HydroLab-3D regression suite still passes
```

with the actual final pytest result recorded rather than predicted.

### Architectural Integrity

```text
learning-specific code remains notebook-local
HydroLab-3D core remains framework-independent
```

unless a genuine production defect is separately demonstrated and corrected.

---

## Expected Final Meaning

If Notebook 11 succeeds, the correct conclusion will **not** be:

> HydroLab-3D is now a reinforcement-learning framework.

The correct conclusion will be:

> **The HydroLab-3D production simulator exposes a sufficiently clean, deterministic, and externally composable interface for a learning algorithm to interact with its existing 6-DOF vehicle model and demonstrate measurable learning without embedding machine-learning machinery into the simulator core.**

Architecturally:

```text
                  HydroLab-3D
                       │
                       │ remains independent
                       ▼
               production Simulator
                       ▲
                       │
         external algorithm interface
                       │
          ┌────────────┼────────────┐
          │            │            │
          ▼            ▼            ▼
      classical      learning      future
      control         policy      autonomy
```

This is the handoff Notebook 11 is intended to validate.

The simulator should emerge from the experiment unchanged in identity:

```text
lightweight
NumPy-first
CPU-friendly
headless-first
transparent
modular
algorithm-first
framework-independent
```

All Notebook 11 learning-specific implementation will remain inside:

```text
11_learning_interface_validation.ipynb
```

and all persistent evidence generated by the notebook will remain inside:

```text
results/notebooks/11_learning_interface_validation/
```

No implementation begins until the inherited project and dependency state are inspected in the first executable step.

## Step 1 - Audit the Inherited Project and Learning Dependencies

Before defining any Gymnasium wrapper or learning task, we first verify the actual HydroLab-3D repository state inherited from Notebook 10.

This step is inspection-only.

It will not:

- modify production source;
- create Notebook 11 artifacts;
- install packages;
- change project dependencies;
- alter simulator configuration.

The audit will verify:

- the detected HydroLab-3D project root;
- presence of `README.md`;
- presence of `src/hydrolab3d/`;
- presence of `tests/`;
- existing notebook result namespaces;
- the current production Python source inventory;
- the current automated-test inventory;
- presence of the production `Simulator`;
- presence of `tests/test_simulator.py`;
- current dependency declarations;
- availability and versions of NumPy and the optional ML stack;
- whether `hydrolab3d.core.Simulator` imports successfully;
- the actual constructor, `reset()`, and `step()` signatures exposed by the current production implementation.

The optional learning packages inspected in this step are:

- `gymnasium`
- `stable-baselines3`
- `torch`

A missing learning package is **not** a HydroLab-3D failure.

If one or more optional ML dependencies are absent, this step will only report that fact. We will diagnose the appropriate dependency action separately before installing or changing anything.

### Validation Criteria

This step passes if:

1. the HydroLab-3D project root is located;
2. the current production source and tests are visible;
3. `hydrolab3d.core.Simulator` imports successfully;
4. its public interface can be inspected;
5. the optional ML dependency state is reported explicitly;
6. no project files are modified.

This gives Notebook 11 an evidence-based starting point before any learning-specific interface is constructed.

In [1]:
from pathlib import Path
import sys
import inspect
import importlib
import importlib.metadata as metadata


# ---------------------------------------------------------------------
# 1. Locate the HydroLab-3D repository root
# ---------------------------------------------------------------------

def find_project_root(start: Path) -> Path:
    start = start.resolve()

    # Search the current directory and its parents first.
    for candidate in [start, *start.parents]:
        if (
            (candidate / "README.md").exists()
            and (candidate / "src" / "hydrolab3d").is_dir()
        ):
            return candidate

    # Fallback: inspect immediate child directories in case Jupyter was
    # launched from one level above the repository.
    for child in start.iterdir():
        if (
            child.is_dir()
            and (child / "README.md").exists()
            and (child / "src" / "hydrolab3d").is_dir()
        ):
            return child.resolve()

    raise FileNotFoundError(
        "Could not locate a HydroLab-3D project root containing both "
        "README.md and src/hydrolab3d/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
TESTS_DIR = PROJECT_ROOT / "tests"
RESULTS_DIR = PROJECT_ROOT / "results" / "notebooks"

print("=" * 72)
print("HYDROLAB-3D NOTEBOOK 11 - INHERITED PROJECT AUDIT")
print("=" * 72)

print(f"\nProject root : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]}")
print(f"Executable   : {sys.executable}")


# ---------------------------------------------------------------------
# 2. Verify important inherited paths
# ---------------------------------------------------------------------

important_paths = {
    "README.md": PROJECT_ROOT / "README.md",
    "src/hydrolab3d": SRC_DIR / "hydrolab3d",
    "tests": TESTS_DIR,
    "results/notebooks": RESULTS_DIR,
    "core/simulator.py": SRC_DIR / "hydrolab3d" / "core" / "simulator.py",
    "core/__init__.py": SRC_DIR / "hydrolab3d" / "core" / "__init__.py",
    "tests/test_simulator.py": TESTS_DIR / "test_simulator.py",
}

print("\n[Important inherited paths]")

for name, path in important_paths.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{status:8s}  {name}")


# ---------------------------------------------------------------------
# 3. Inventory the current production source tree
# ---------------------------------------------------------------------

print("\n[Production source files]")

hydrolab_package = SRC_DIR / "hydrolab3d"

source_files = sorted(
    p.relative_to(PROJECT_ROOT)
    for p in hydrolab_package.rglob("*.py")
)

for path in source_files:
    print(f"  {path}")


# ---------------------------------------------------------------------
# 4. Inventory the current automated tests
# ---------------------------------------------------------------------

print("\n[Test files]")

if TESTS_DIR.exists():
    test_files = sorted(
        p.relative_to(PROJECT_ROOT)
        for p in TESTS_DIR.glob("test_*.py")
    )

    for path in test_files:
        print(f"  {path}")
else:
    print("  tests/ directory not found")


# ---------------------------------------------------------------------
# 5. Inspect existing notebook result namespaces
# ---------------------------------------------------------------------

print("\n[Existing notebook result directories]")

if RESULTS_DIR.exists():
    result_dirs = sorted(
        p.name
        for p in RESULTS_DIR.iterdir()
        if p.is_dir()
    )

    for name in result_dirs:
        print(f"  {name}")
else:
    print("  results/notebooks/ does not currently exist")


# ---------------------------------------------------------------------
# 6. Inspect declared project dependencies
# ---------------------------------------------------------------------

print("\n[Dependency declaration files]")

requirements_path = PROJECT_ROOT / "requirements.txt"
pyproject_path = PROJECT_ROOT / "pyproject.toml"

if requirements_path.exists():
    print("\n--- requirements.txt ---")
    requirements_text = requirements_path.read_text(encoding="utf-8").strip()

    if requirements_text:
        print(requirements_text)
    else:
        print("(empty)")
else:
    print("requirements.txt : not present")


if pyproject_path.exists():
    print("\n--- pyproject.toml ---")

    pyproject_lines = pyproject_path.read_text(
        encoding="utf-8"
    ).splitlines()

    for line in pyproject_lines[:120]:
        print(line)

    if len(pyproject_lines) > 120:
        print(
            f"... ({len(pyproject_lines) - 120} additional lines not shown)"
        )
else:
    print("pyproject.toml    : not present")


# ---------------------------------------------------------------------
# 7. Audit relevant installed packages
# ---------------------------------------------------------------------

print("\n[Relevant installed packages]")

packages = {
    "numpy": "numpy",
    "gymnasium": "gymnasium",
    "stable-baselines3": "stable_baselines3",
    "torch": "torch",
}

package_status = {}

for distribution_name, import_name in packages.items():

    try:
        version = metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        version = None

    try:
        importlib.import_module(import_name)
        import_ok = True
        import_error = None
    except Exception as exc:
        import_ok = False
        import_error = f"{type(exc).__name__}: {exc}"

    package_status[distribution_name] = {
        "version": version,
        "import_ok": import_ok,
        "error": import_error,
    }

    if version is None:
        print(f"{distribution_name:20s} : NOT INSTALLED")

    elif import_ok:
        print(
            f"{distribution_name:20s} : "
            f"{version}  [import OK]"
        )

    else:
        print(
            f"{distribution_name:20s} : "
            f"{version}  [import FAILED: {import_error}]"
        )


# ---------------------------------------------------------------------
# 8. Import the production HydroLab-3D Simulator
# ---------------------------------------------------------------------

src_string = str(SRC_DIR)

if src_string not in sys.path:
    sys.path.insert(0, src_string)

from hydrolab3d.core import Simulator


# ---------------------------------------------------------------------
# 9. Inspect the actual public Simulator interface
# ---------------------------------------------------------------------

print("\n[Production Simulator interface]")

print(f"Simulator module : {Simulator.__module__}")
print(f"Simulator class  : {Simulator.__qualname__}")

print("\nConstructor:")
print(f"  Simulator{inspect.signature(Simulator)}")

print("\nreset():")
print(f"  {inspect.signature(Simulator.reset)}")

print("\nstep():")
print(f"  {inspect.signature(Simulator.step)}")


# ---------------------------------------------------------------------
# 10. Notebook 11 readiness summary
# ---------------------------------------------------------------------

print("\n" + "=" * 72)
print("NOTEBOOK 11 READINESS SUMMARY")
print("=" * 72)

required_core_keys = (
    "README.md",
    "src/hydrolab3d",
    "tests",
    "core/simulator.py",
    "core/__init__.py",
    "tests/test_simulator.py",
)

core_ready = all(
    important_paths[key].exists()
    for key in required_core_keys
)

ml_ready = all(
    package_status[name]["import_ok"]
    for name in (
        "gymnasium",
        "stable-baselines3",
        "torch",
    )
)

print(
    "Production project structure : "
    f"{'PASS' if core_ready else 'CHECK REQUIRED'}"
)

print(
    "Simulator import             : PASS"
)

print(
    "Optional ML stack available  : "
    f"{'YES' if ml_ready else 'NO / PARTIAL'}"
)

print("\nNo files were modified and no packages were installed.")

HYDROLAB-3D NOTEBOOK 11 - INHERITED PROJECT AUDIT

Project root : /home/agniacolyte/HydroLab-3D
Python       : 3.12.3
Executable   : /home/agniacolyte/HydroLab-3D/.venv/bin/python

[Important inherited paths]
FOUND     README.md
FOUND     src/hydrolab3d
FOUND     tests
FOUND     results/notebooks
FOUND     core/simulator.py
FOUND     core/__init__.py
FOUND     tests/test_simulator.py

[Production source files]
  src/hydrolab3d/__init__.py
  src/hydrolab3d/communication/__init__.py
  src/hydrolab3d/communication/acoustic_channel.py
  src/hydrolab3d/communication/network.py
  src/hydrolab3d/communication/packet.py
  src/hydrolab3d/control/__init__.py
  src/hydrolab3d/control/controller.py
  src/hydrolab3d/control/pid.py
  src/hydrolab3d/control/waypoint.py
  src/hydrolab3d/core/__init__.py
  src/hydrolab3d/core/clock.py
  src/hydrolab3d/core/entity.py
  src/hydrolab3d/core/simulator.py
  src/hydrolab3d/core/state.py
  src/hydrolab3d/dynamics/__init__.py
  src/hydrolab3d/dynamics/buoyancy

## Step 2 - Validate the Actual Production Simulator Runtime Contract

Step 1 confirmed that the production `Simulator` and the complete optional learning stack are available.

The first attempt at this step tried to recover configuration variables by executing selected Notebook 10 cells independently. That approach was invalid because those cells depend on earlier notebook variables such as the validated mass, inertia, damping, and hydrostatic parameters.

We will not reconstruct those dependencies or guess missing values.

Instead, this replacement step uses the **self-contained production configuration already used by Notebook 10 for its fresh-process validation**.

That configuration was explicitly designed to prove that a completely separate Python interpreter could import and execute:

`hydrolab3d.core.Simulator`

without relying on notebook-local simulator classes or hidden notebook state.

The validated configuration uses:

- mass: 50 kg;
- inertia: `[2, 3, 4]`;
- added inertia: `[5, 8, 10, 0.5, 0.7, 0.9]`;
- linear damping: `[8, 12, 15, 1.5, 2, 2.5]`;
- quadratic damping: `[12, 18, 22, 2, 2.5, 3]`;
- seawater density: 1025 kg/m³;
- neutral displaced volume derived from mass and fluid density;
- center of gravity at the body origin;
- center of buoyancy 0.08 m above the body origin;
- uniform world-frame current `[0.15, -0.05, 0.0]` m/s;
- fixed timestep `0.02 s`;
- the same explicit sensor-noise configuration used in Notebook 10's production validation.

This step performs no learning and defines no Gymnasium wrapper.

It will inspect three distinct production interfaces:

### Reset contract

`Simulator.reset()` should return ground truth only:

- simulation time;
- six-element pose;
- six-element body velocity.

### Step contract

`Simulator.step(action)` should accept the production generalized body-frame wrench:

`[Fx, Fy, Fz, Mx, My, Mz]`

and return:

- `truth`;
- `observation`.

The truth branch should contain:

- time;
- pose;
- body velocity.

The observation branch should contain:

- depth;
- heading;
- DVL;
- gyroscope;
- accelerometer.

### Runtime behavior

The cell will perform:

1. one reset;
2. one zero-wrench step;
3. one representative translational-wrench step after another reset.

The representative action is intentionally modest relative to actions already exercised in Notebook 10.

No source files, tests, dependency files, or result artifacts are modified.

### Validation Criteria

This step passes if:

1. the production `Simulator` can be instantiated from the validated Notebook 10 configuration;
2. `reset()` returns finite six-degree-of-freedom truth state;
3. `step()` accepts a six-element generalized wrench;
4. the returned `truth` and `observation` structures match the inherited production contract;
5. pose and body velocity remain shape `(6,)`;
6. DVL, gyroscope, and accelerometer remain shape `(3,)`;
7. all returned numerical values are finite;
8. simulation time advances exactly one configured timestep;
9. reset restores time and physical state exactly;
10. no project files are modified.

The resulting runtime contract will become the foundation for the Notebook 11 learning environment in the next step.

In [3]:
import numpy as np

from hydrolab3d.core import Simulator


# ---------------------------------------------------------------------
# 1. Reuse the self-contained production configuration validated by
#    Notebook 10 in its fresh-process execution test.
# ---------------------------------------------------------------------

contract_initial_pose = np.array(
    [
        0.0,
        0.0,
        -5.0,
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)

contract_initial_body_velocity = np.zeros(
    6,
    dtype=float,
)

contract_dt = 0.02


# ---------------------------------------------------------------------
# Validated physical parameters
# ---------------------------------------------------------------------

contract_mass = 50.0

contract_inertia = np.array(
    [
        2.0,
        3.0,
        4.0,
    ],
    dtype=float,
)

contract_added_inertia = np.array(
    [
        5.0,
        8.0,
        10.0,
        0.5,
        0.7,
        0.9,
    ],
    dtype=float,
)

contract_linear_damping = np.array(
    [
        8.0,
        12.0,
        15.0,
        1.5,
        2.0,
        2.5,
    ],
    dtype=float,
)

contract_quadratic_damping = np.array(
    [
        12.0,
        18.0,
        22.0,
        2.0,
        2.5,
        3.0,
    ],
    dtype=float,
)

contract_fluid_density = 1025.0

contract_displaced_volume = (
    contract_mass
    / contract_fluid_density
)

contract_dynamics_kwargs = {
    "mass": contract_mass,
    "inertia": contract_inertia,
    "added_inertia": contract_added_inertia,
    "linear_damping": contract_linear_damping,
    "quadratic_damping": contract_quadratic_damping,
    "fluid_density": contract_fluid_density,
    "displaced_volume": contract_displaced_volume,
    "center_of_gravity": np.array(
        [
            0.0,
            0.0,
            0.0,
        ],
        dtype=float,
    ),
    "center_of_buoyancy": np.array(
        [
            0.0,
            0.0,
            0.08,
        ],
        dtype=float,
    ),
    "current_velocity_world": np.array(
        [
            0.15,
            -0.05,
            0.0,
        ],
        dtype=float,
    ),
}


# ---------------------------------------------------------------------
# 2. Reuse Notebook 10's explicit production sensor configuration
# ---------------------------------------------------------------------

contract_sensor_config = {
    "depth": {
        "bias": 0.03,
        "noise_std": 0.04,
    },
    "heading": {
        "bias": np.deg2rad(0.75),
        "noise_std": np.deg2rad(1.0),
    },
    "dvl": {
        "bias": np.array(
            [
                0.01,
                -0.01,
                0.005,
            ],
            dtype=float,
        ),
        "noise_std": 0.015,
    },
    "gyroscope": {
        "bias": np.array(
            [
                0.001,
                -0.001,
                0.0005,
            ],
            dtype=float,
        ),
        "noise_std": 0.002,
    },
    "accelerometer": {
        "bias": np.array(
            [
                0.02,
                -0.01,
                0.03,
            ],
            dtype=float,
        ),
        "noise_std": 0.025,
    },
}

contract_sensor_seed = 1111


# ---------------------------------------------------------------------
# 3. Construct the actual production Simulator
# ---------------------------------------------------------------------

contract_simulator = Simulator(
    initial_pose=contract_initial_pose,
    initial_body_velocity=(
        contract_initial_body_velocity
    ),
    dt=contract_dt,
    dynamics_kwargs=(
        contract_dynamics_kwargs
    ),
    sensor_config=(
        contract_sensor_config
    ),
    sensor_seed=contract_sensor_seed,
)


# ---------------------------------------------------------------------
# 4. Helper functions for strict structural validation
# ---------------------------------------------------------------------

def assert_finite_array(
    value,
    expected_shape,
    name,
):
    value = np.asarray(
        value,
        dtype=float,
    )

    assert value.shape == expected_shape, (
        f"{name} has shape {value.shape}; "
        f"expected {expected_shape}"
    )

    assert np.all(
        np.isfinite(value)
    ), (
        f"{name} contains NaN or Inf"
    )

    return value


def validate_truth(
    truth,
    expected_time,
):
    assert isinstance(
        truth,
        dict,
    )

    assert set(
        truth.keys()
    ) == {
        "time",
        "pose",
        "body_velocity",
    }

    assert np.isfinite(
        truth["time"]
    )

    assert np.isclose(
        truth["time"],
        expected_time,
        rtol=0.0,
        atol=1e-15,
    )

    pose = assert_finite_array(
        truth["pose"],
        (6,),
        "truth['pose']",
    )

    body_velocity = assert_finite_array(
        truth["body_velocity"],
        (6,),
        "truth['body_velocity']",
    )

    return pose, body_velocity


def validate_observation(
    observation,
):
    assert isinstance(
        observation,
        dict,
    )

    expected_keys = {
        "depth",
        "heading",
        "dvl",
        "gyroscope",
        "accelerometer",
    }

    assert set(
        observation.keys()
    ) == expected_keys

    assert np.isfinite(
        observation["depth"]
    )

    assert np.isfinite(
        observation["heading"]
    )

    dvl = assert_finite_array(
        observation["dvl"],
        (3,),
        "observation['dvl']",
    )

    gyroscope = assert_finite_array(
        observation["gyroscope"],
        (3,),
        "observation['gyroscope']",
    )

    accelerometer = assert_finite_array(
        observation["accelerometer"],
        (3,),
        "observation['accelerometer']",
    )

    return (
        dvl,
        gyroscope,
        accelerometer,
    )


# ---------------------------------------------------------------------
# 5. Validate reset() contract
# ---------------------------------------------------------------------

reset_state = (
    contract_simulator.reset()
)

reset_pose, reset_velocity = (
    validate_truth(
        reset_state,
        expected_time=0.0,
    )
)

assert np.array_equal(
    reset_pose,
    contract_initial_pose,
)

assert np.array_equal(
    reset_velocity,
    contract_initial_body_velocity,
)


# ---------------------------------------------------------------------
# 6. Validate one zero-wrench step
# ---------------------------------------------------------------------

zero_action = np.zeros(
    6,
    dtype=float,
)

zero_result = (
    contract_simulator.step(
        zero_action
    )
)

assert isinstance(
    zero_result,
    dict,
)

assert set(
    zero_result.keys()
) == {
    "truth",
    "observation",
}

zero_pose, zero_velocity = (
    validate_truth(
        zero_result["truth"],
        expected_time=contract_dt,
    )
)

(
    zero_dvl,
    zero_gyro,
    zero_accel,
) = validate_observation(
    zero_result["observation"]
)


# ---------------------------------------------------------------------
# 7. Reset again and verify exact reset semantics
# ---------------------------------------------------------------------

second_reset_state = (
    contract_simulator.reset()
)

second_reset_pose, second_reset_velocity = (
    validate_truth(
        second_reset_state,
        expected_time=0.0,
    )
)

assert np.array_equal(
    second_reset_pose,
    contract_initial_pose,
)

assert np.array_equal(
    second_reset_velocity,
    contract_initial_body_velocity,
)


# ---------------------------------------------------------------------
# 8. Validate one representative translational-wrench step
#
# This does not define the eventual learning-action scale.
# It is only a production-interface probe.
# ---------------------------------------------------------------------

test_action = np.array(
    [
        10.0,
        2.0,
        -2.0,
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)

test_action_reference = (
    test_action.copy()
)

test_result = (
    contract_simulator.step(
        test_action
    )
)

assert set(
    test_result.keys()
) == {
    "truth",
    "observation",
}

test_pose, test_velocity = (
    validate_truth(
        test_result["truth"],
        expected_time=contract_dt,
    )
)

(
    test_dvl,
    test_gyro,
    test_accel,
) = validate_observation(
    test_result["observation"]
)


# ---------------------------------------------------------------------
# 9. Confirm caller-owned action was not modified
# ---------------------------------------------------------------------

assert np.array_equal(
    test_action,
    test_action_reference,
)


# ---------------------------------------------------------------------
# 10. Explicitly verify time property
# ---------------------------------------------------------------------

assert np.isclose(
    contract_simulator.time,
    contract_dt,
    rtol=0.0,
    atol=1e-15,
)

assert np.isclose(
    contract_simulator.dt,
    contract_dt,
    rtol=0.0,
    atol=0.0,
)


# ---------------------------------------------------------------------
# 11. Concise production-contract report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 11 - STEP 2 PRODUCTION CONTRACT")
print("=" * 80)


print("\nProduction configuration")
print("-" * 80)

print(
    f"mass              : "
    f"{contract_mass:.3f} kg"
)

print(
    f"dt                : "
    f"{contract_dt:.6f} s"
)

print(
    "initial pose      :",
    contract_initial_pose,
)

print(
    "initial velocity  :",
    contract_initial_body_velocity,
)

print(
    "world current     :",
    contract_dynamics_kwargs[
        "current_velocity_world"
    ],
)

print(
    f"sensor seed       : "
    f"{contract_sensor_seed}"
)


print("\nreset() contract")
print("-" * 80)

print(
    "keys              :",
    sorted(
        reset_state.keys()
    ),
)

print(
    f"time              : "
    f"{reset_state['time']:.6f} s"
)

print(
    "pose shape        :",
    reset_state["pose"].shape,
)

print(
    "velocity shape    :",
    reset_state[
        "body_velocity"
    ].shape,
)

print(
    "exact initial pose:",
    np.array_equal(
        reset_state["pose"],
        contract_initial_pose,
    ),
)

print(
    "exact initial vel :",
    np.array_equal(
        reset_state[
            "body_velocity"
        ],
        contract_initial_body_velocity,
    ),
)


print("\nstep() top-level contract")
print("-" * 80)

print(
    "keys              :",
    sorted(
        test_result.keys()
    ),
)

print(
    "truth keys        :",
    sorted(
        test_result[
            "truth"
        ].keys()
    ),
)

print(
    "observation keys  :",
    sorted(
        test_result[
            "observation"
        ].keys()
    ),
)


print("\nRepresentative test action")
print("-" * 80)

print(
    "body wrench       :",
    test_action,
)

print(
    "action preserved  :",
    np.array_equal(
        test_action,
        test_action_reference,
    ),
)


print("\nPost-step truth")
print("-" * 80)

print(
    f"time              : "
    f"{test_result['truth']['time']:.6f} s"
)

print(
    "pose              :",
    test_pose,
)

print(
    "body velocity     :",
    test_velocity,
)


print("\nPost-step observation")
print("-" * 80)

print(
    f"depth             : "
    f"{test_result['observation']['depth']:.9f}"
)

print(
    f"heading           : "
    f"{test_result['observation']['heading']:.9f}"
)

print(
    "DVL               :",
    test_dvl,
)

print(
    "gyroscope         :",
    test_gyro,
)

print(
    "accelerometer     :",
    test_accel,
)


print("\nContract validation")
print("-" * 80)

print(
    "reset truth finite             : True"
)

print(
    "zero-wrench step finite        : True"
)

print(
    "translational test step finite : True"
)

print(
    "truth pose shape               : (6,)"
)

print(
    "truth velocity shape           : (6,)"
)

print(
    "DVL shape                      : (3,)"
)

print(
    "gyroscope shape                : (3,)"
)

print(
    "accelerometer shape            : (3,)"
)

print(
    "time advances exactly by dt    :",
    np.isclose(
        test_result[
            "truth"
        ]["time"],
        contract_dt,
        rtol=0.0,
        atol=1e-15,
    ),
)


print("\n" + "=" * 80)
print("STEP 2 PRODUCTION SIMULATOR CONTRACT PASSED")
print("=" * 80)

print(
    "The production Simulator was instantiated using the "
    "self-contained configuration already validated by Notebook 10."
)

print(
    "reset() exposes ground truth, while step() returns separate "
    "truth and post-step sensor-observation branches."
)

print(
    "The production action interface is confirmed as a finite "
    "six-element generalized body-frame wrench."
)

print(
    "No source files, tests, dependencies, or result artifacts "
    "were modified."
)

HYDROLAB-3D NOTEBOOK 11 - STEP 2 PRODUCTION CONTRACT

Production configuration
--------------------------------------------------------------------------------
mass              : 50.000 kg
dt                : 0.020000 s
initial pose      : [ 0.  0. -5.  0.  0.  0.]
initial velocity  : [0. 0. 0. 0. 0. 0.]
world current     : [ 0.15 -0.05  0.  ]
sensor seed       : 1111

reset() contract
--------------------------------------------------------------------------------
keys              : ['body_velocity', 'pose', 'time']
time              : 0.000000 s
pose shape        : (6,)
velocity shape    : (6,)
exact initial pose: True
exact initial vel : True

step() top-level contract
--------------------------------------------------------------------------------
keys              : ['observation', 'truth']
truth keys        : ['body_velocity', 'pose', 'time']
observation keys  : ['accelerometer', 'depth', 'dvl', 'gyroscope', 'heading']

Representative test action
-------------------------------

## Step 3 - Define and Validate the Notebook-Local Learning Task Contract

With the production `Simulator` interface now validated, we can define the smallest learning task required for Notebook 11.

This step remains **notebook-local** and does not introduce a Gymnasium environment yet.

The task will use:

- the production initial state validated in Step 2;
- a fixed 3D waypoint;
- truth-derived policy observations;
- a normalized three-dimensional learning action;
- explicit mapping from that learning action into the production six-dimensional body-frame wrench;
- a progress-based reward;
- a finite success radius;
- a fixed episode horizon.

### Observation

The policy observation is:

$$
o_t =
[
\Delta x,
\Delta y,
\Delta z,
\phi,
\theta,
\psi,
u,
v,
w,
p,
q,
r
]^T
$$

where the first three terms are the **world-frame vector from the vehicle to the goal**.

The remaining nine terms come directly from production simulator truth:

- roll, pitch, yaw;
- body-frame linear velocity;
- body-frame angular velocity.

This produces a 12-element observation.

This is deliberately a **truth-state smoke test**, not a sensor-based navigation claim.

### Learning Action

The learning action is:

$$
a_t = [a_x,a_y,a_z]^T,
\qquad
a_i \in [-1,1].
$$

It is mapped into the production generalized wrench as:

$$
\tau_t =
[
F_{x,\max}a_x,
F_{y,\max}a_y,
F_{z,\max}a_z,
0,
0,
0
]^T.
$$

For this smoke test we use a translational force scale of:

$$
[F_{x,\max},F_{y,\max},F_{z,\max}]
=
[20,20,20]\ \text{N}.
$$

This magnitude is consistent with the force scale already exercised successfully in Notebook 10 and remains modest for the validated 50 kg vehicle.

### Waypoint Task

The fixed goal is:

$$
\mathbf{p}_{goal}
=
[6,-2,-4]^T\ \text{m}.
$$

From the inherited initial position:

$$
[0,0,-5]^T,
$$

the initial goal distance is approximately:

$$
\sqrt{6^2+(-2)^2+1^2}
\approx 6.40\ \text{m}.
$$

This creates a nontrivial three-axis task while keeping the smoke test compact.

### Reward

The notebook-local reward is:

$$
r_t =
5(d_{t-1}-d_t)
-
0.01\|a_t\|_2^2
-
0.01
+
r_{success}.
$$

where:

- progress toward the goal earns positive reward;
- movement away from the goal produces negative progress;
- excessive action receives a small penalty;
- every step receives a small time penalty;
- entering the success region gives an additional bonus of `+25`.

Success is defined by:

$$
d_t \le 0.50\ \text{m}.
$$

The maximum episode length is initially:

$$
N_{max}=750
$$

steps, corresponding to:

$$
750(0.02)=15\ \text{s}
$$

of simulated time.

These parameters are part of the temporary learning task and do not modify HydroLab-3D physics.

### Validation Criteria

This step passes if:

1. the observation has shape `(12,)`;
2. all observation values are finite;
3. the initial relative-goal vector is correct;
4. normalized actions are clipped to `[-1,1]`;
5. the mapped production wrench has shape `(6,)`;
6. rotational wrench components remain exactly zero;
7. the force mapping respects the declared ±20 N limits;
8. one production step using the mapped action remains finite;
9. reward calculation produces a finite scalar;
10. success and horizon definitions are represented explicitly.

No Gymnasium wrapper, PPO model, artifact, or production source change is introduced yet.

In [5]:
import numpy as np


# ---------------------------------------------------------------------
# 1. Notebook-local learning-task constants
# ---------------------------------------------------------------------

LEARNING_GOAL_POSITION = np.array(
    [
        6.0,
        -2.0,
        -4.0,
    ],
    dtype=float,
)

LEARNING_FORCE_LIMITS = np.array(
    [
        20.0,
        20.0,
        20.0,
    ],
    dtype=float,
)

LEARNING_SUCCESS_RADIUS = 0.50

LEARNING_MAX_STEPS = 750

LEARNING_PROGRESS_WEIGHT = 5.0

LEARNING_ACTION_PENALTY = 0.01

LEARNING_STEP_PENALTY = 0.01

LEARNING_SUCCESS_BONUS = 25.0


# ---------------------------------------------------------------------
# 2. Observation constructor
# ---------------------------------------------------------------------

def build_learning_observation(
    truth,
    goal_position,
):
    """
    Construct the Notebook 11 truth-derived learning observation.

    Observation:
        [
            dx, dy, dz,
            roll, pitch, yaw,
            u, v, w,
            p, q, r,
        ]

    Goal displacement is expressed in the world frame.
    Velocity remains in the production body frame.
    """

    pose = np.asarray(
        truth["pose"],
        dtype=float,
    )

    body_velocity = np.asarray(
        truth["body_velocity"],
        dtype=float,
    )

    goal_position = np.asarray(
        goal_position,
        dtype=float,
    )

    if pose.shape != (6,):
        raise ValueError(
            "truth pose must have shape (6,)"
        )

    if body_velocity.shape != (6,):
        raise ValueError(
            "truth body_velocity must have shape (6,)"
        )

    if goal_position.shape != (3,):
        raise ValueError(
            "goal_position must have shape (3,)"
        )

    relative_goal = (
        goal_position
        - pose[:3]
    )

    observation = np.concatenate(
        [
            relative_goal,
            pose[3:6],
            body_velocity,
        ]
    ).astype(
        np.float32,
        copy=False,
    )

    if observation.shape != (12,):
        raise RuntimeError(
            "learning observation must have shape (12,)"
        )

    if not np.all(
        np.isfinite(
            observation
        )
    ):
        raise FloatingPointError(
            "learning observation contains NaN or Inf"
        )

    return observation


# ---------------------------------------------------------------------
# 3. Learning-action to production-wrench mapping
# ---------------------------------------------------------------------

def map_learning_action_to_wrench(
    learning_action,
    force_limits,
):
    """
    Convert a normalized 3D learning action into the production
    six-dimensional generalized body-frame wrench.
    """

    learning_action = np.asarray(
        learning_action,
        dtype=float,
    )

    force_limits = np.asarray(
        force_limits,
        dtype=float,
    )

    if learning_action.shape != (3,):
        raise ValueError(
            "learning_action must have shape (3,)"
        )

    if force_limits.shape != (3,):
        raise ValueError(
            "force_limits must have shape (3,)"
        )

    if not np.all(
        np.isfinite(
            learning_action
        )
    ):
        raise ValueError(
            "learning_action must contain only finite values"
        )

    if not np.all(
        np.isfinite(
            force_limits
        )
    ):
        raise ValueError(
            "force_limits must contain only finite values"
        )

    if not np.all(
        force_limits > 0.0
    ):
        raise ValueError(
            "force_limits must be strictly positive"
        )

    clipped_action = np.clip(
        learning_action,
        -1.0,
        1.0,
    )

    translational_force = (
        clipped_action
        * force_limits
    )

    wrench = np.array(
        [
            translational_force[0],
            translational_force[1],
            translational_force[2],
            0.0,
            0.0,
            0.0,
        ],
        dtype=float,
    )

    return (
        clipped_action,
        wrench,
    )


# ---------------------------------------------------------------------
# 4. Distance helper
# ---------------------------------------------------------------------

def learning_goal_distance(
    truth,
    goal_position,
):
    position = np.asarray(
        truth["pose"][:3],
        dtype=float,
    )

    goal_position = np.asarray(
        goal_position,
        dtype=float,
    )

    distance = float(
        np.linalg.norm(
            goal_position
            - position
        )
    )

    if not np.isfinite(
        distance
    ):
        raise FloatingPointError(
            "goal distance is not finite"
        )

    return distance


# ---------------------------------------------------------------------
# 5. Reward helper
# ---------------------------------------------------------------------

def calculate_learning_reward(
    previous_distance,
    current_distance,
    normalized_action,
    *,
    success,
):
    """
    Notebook-local progress reward.

    Positive reward:
        movement toward goal

    Penalties:
        action magnitude
        elapsed step

    Bonus:
        successful goal entry
    """

    normalized_action = np.asarray(
        normalized_action,
        dtype=float,
    )

    progress = (
        previous_distance
        - current_distance
    )

    action_cost = float(
        np.dot(
            normalized_action,
            normalized_action,
        )
    )

    reward = (
        LEARNING_PROGRESS_WEIGHT
        * progress
        - LEARNING_ACTION_PENALTY
        * action_cost
        - LEARNING_STEP_PENALTY
    )

    if success:
        reward += (
            LEARNING_SUCCESS_BONUS
        )

    reward = float(
        reward
    )

    if not np.isfinite(
        reward
    ):
        raise FloatingPointError(
            "learning reward is not finite"
        )

    return reward


# ---------------------------------------------------------------------
# 6. Reset production Simulator and construct initial observation
# ---------------------------------------------------------------------

task_initial_truth = (
    contract_simulator.reset()
)

task_initial_observation = (
    build_learning_observation(
        task_initial_truth,
        LEARNING_GOAL_POSITION,
    )
)

task_initial_distance = (
    learning_goal_distance(
        task_initial_truth,
        LEARNING_GOAL_POSITION,
    )
)


# ---------------------------------------------------------------------
# 7. Validate initial observation analytically
# ---------------------------------------------------------------------

expected_relative_goal = np.array(
    [
        6.0,
        -2.0,
        1.0,
    ],
    dtype=float,
)

assert (
    task_initial_observation.shape
    == (12,)
)

assert (
    task_initial_observation.dtype
    == np.float32
)

assert np.all(
    np.isfinite(
        task_initial_observation
    )
)

assert np.allclose(
    task_initial_observation[:3],
    expected_relative_goal,
    rtol=0.0,
    atol=1e-7,
)

expected_initial_distance = float(
    np.sqrt(
        6.0**2
        + (-2.0)**2
        + 1.0**2
    )
)

assert np.isclose(
    task_initial_distance,
    expected_initial_distance,
    rtol=0.0,
    atol=1e-12,
)


# ---------------------------------------------------------------------
# 8. Probe the action mapper
#
# Deliberately include one value outside [-1, 1] to verify clipping.
# ---------------------------------------------------------------------

task_probe_action = np.array(
    [
        0.50,
        -1.25,
        0.25,
    ],
    dtype=float,
)

(
    task_clipped_action,
    task_wrench,
) = map_learning_action_to_wrench(
    task_probe_action,
    LEARNING_FORCE_LIMITS,
)

expected_clipped_action = np.array(
    [
        0.50,
        -1.00,
        0.25,
    ],
    dtype=float,
)

expected_wrench = np.array(
    [
        10.0,
        -20.0,
        5.0,
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)

assert np.array_equal(
    task_clipped_action,
    expected_clipped_action,
)

assert np.array_equal(
    task_wrench,
    expected_wrench,
)

assert (
    task_wrench.shape
    == (6,)
)

assert np.array_equal(
    task_wrench[3:],
    np.zeros(
        3,
        dtype=float,
    ),
)

assert np.all(
    np.abs(
        task_wrench[:3]
    )
    <= LEARNING_FORCE_LIMITS
)


# ---------------------------------------------------------------------
# 9. Advance one real production-simulator step through mapped action
# ---------------------------------------------------------------------

task_step_result = (
    contract_simulator.step(
        task_wrench
    )
)

task_next_truth = (
    task_step_result["truth"]
)

task_next_observation = (
    build_learning_observation(
        task_next_truth,
        LEARNING_GOAL_POSITION,
    )
)

task_next_distance = (
    learning_goal_distance(
        task_next_truth,
        LEARNING_GOAL_POSITION,
    )
)


# ---------------------------------------------------------------------
# 10. Evaluate success and reward
# ---------------------------------------------------------------------

task_success = bool(
    task_next_distance
    <= LEARNING_SUCCESS_RADIUS
)

task_reward = (
    calculate_learning_reward(
        task_initial_distance,
        task_next_distance,
        task_clipped_action,
        success=task_success,
    )
)


# ---------------------------------------------------------------------
# 11. Validate resulting transition
# ---------------------------------------------------------------------

assert (
    task_next_observation.shape
    == (12,)
)

assert np.all(
    np.isfinite(
        task_next_observation
    )
)

assert np.all(
    np.isfinite(
        task_next_truth["pose"]
    )
)

assert np.all(
    np.isfinite(
        task_next_truth[
            "body_velocity"
        ]
    )
)

assert np.isfinite(
    task_next_distance
)

assert np.isfinite(
    task_reward
)

assert (
    LEARNING_MAX_STEPS
    * contract_dt
    == 15.0
)


# ---------------------------------------------------------------------
# 12. Concise task-contract report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 11 - STEP 3 LEARNING TASK CONTRACT")
print("=" * 80)


print("\nTask geometry")
print("-" * 80)

print(
    "initial position        :",
    task_initial_truth[
        "pose"
    ][:3],
)

print(
    "goal position           :",
    LEARNING_GOAL_POSITION,
)

print(
    "initial relative goal   :",
    task_initial_observation[
        :3
    ],
)

print(
    f"initial goal distance   : "
    f"{task_initial_distance:.9f} m"
)

print(
    f"success radius          : "
    f"{LEARNING_SUCCESS_RADIUS:.3f} m"
)


print("\nObservation contract")
print("-" * 80)

print(
    "shape                   :",
    task_initial_observation.shape,
)

print(
    "dtype                   :",
    task_initial_observation.dtype,
)

print(
    "observation             :",
    task_initial_observation,
)

print(
    "all finite              :",
    np.all(
        np.isfinite(
            task_initial_observation
        )
    ),
)


print("\nAction contract")
print("-" * 80)

print(
    "raw normalized action   :",
    task_probe_action,
)

print(
    "clipped action          :",
    task_clipped_action,
)

print(
    "force limits [N]        :",
    LEARNING_FORCE_LIMITS,
)

print(
    "production wrench       :",
    task_wrench,
)

print(
    "rotational moments zero :",
    np.array_equal(
        task_wrench[3:],
        np.zeros(3),
    ),
)


print("\nOne-step transition")
print("-" * 80)

print(
    f"simulation time         : "
    f"{task_next_truth['time']:.6f} s"
)

print(
    "next position           :",
    task_next_truth[
        "pose"
    ][:3],
)

print(
    "next body velocity      :",
    task_next_truth[
        "body_velocity"
    ],
)

print(
    f"next goal distance      : "
    f"{task_next_distance:.9f} m"
)

print(
    f"distance progress       : "
    f"{task_initial_distance - task_next_distance:.12f} m"
)

print(
    f"reward                  : "
    f"{task_reward:.12f}"
)

print(
    f"success                 : "
    f"{task_success}"
)


print("\nEpisode contract")
print("-" * 80)

print(
    f"maximum steps           : "
    f"{LEARNING_MAX_STEPS}"
)

print(
    f"maximum simulated time  : "
    f"{LEARNING_MAX_STEPS * contract_dt:.3f} s"
)

print(
    f"success bonus           : "
    f"{LEARNING_SUCCESS_BONUS:.3f}"
)


print("\n" + "=" * 80)
print("STEP 3 LEARNING TASK CONTRACT PASSED")
print("=" * 80)

print(
    "The notebook-local learning task now has an explicit "
    "12-element truth-derived observation."
)

print(
    "Normalized 3D learning actions map explicitly into the "
    "production six-axis body-frame wrench without changing "
    "the Simulator action contract."
)

print(
    "Goal geometry, reward, success radius, and episode horizon "
    "are now explicit and ready for Gymnasium wrapping."
)

HYDROLAB-3D NOTEBOOK 11 - STEP 3 LEARNING TASK CONTRACT

Task geometry
--------------------------------------------------------------------------------
initial position        : [ 0.  0. -5.]
goal position           : [ 6. -2. -4.]
initial relative goal   : [ 6. -2.  1.]
initial goal distance   : 6.403124237 m
success radius          : 0.500 m

Observation contract
--------------------------------------------------------------------------------
shape                   : (12,)
dtype                   : float32
observation             : [ 6. -2.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
all finite              : True

Action contract
--------------------------------------------------------------------------------
raw normalized action   : [ 0.5  -1.25  0.25]
clipped action          : [ 0.5  -1.    0.25]
force limits [N]        : [20. 20. 20.]
production wrench       : [ 10. -20.   5.   0.   0.   0.]
rotational moments zero : True

One-step transition
--------------------------------------

## Step 4 - Build and Validate the Temporary Gymnasium Environment

Step 3 established the complete notebook-local learning-task contract.

This step wraps that contract in a temporary Gymnasium environment so Stable-Baselines3 can interact with the existing production `hydrolab3d.core.Simulator`.

The first Step 4 attempt reached the production-wrench validation successfully but used exact floating-point equality when comparing a wrench generated from a `float32` Gymnasium action.

Gymnasium actions use `float32`, so values such as:

    0.10 × 20.0

are not guaranteed to equal the Python `float64` value `2.0` bit-for-bit.

That is ordinary floating-point representation behavior, not an action-mapping defect.

The replacement therefore preserves the environment architecture and changes the numerical validation to tolerance-based comparison where floating-point arithmetic is involved.

No production HydroLab-3D code is changed.

### Environment Architecture

The notebook-local wrapper performs:

    normalized 3D learning action
                ↓
        explicit action mapping
                ↓
    [Fx, Fy, Fz, 0, 0, 0]
                ↓
      production Simulator.step()
                ↓
        production truth state
                ↓
      12D learning observation
                ↓
     reward + episode bookkeeping

### Gymnasium Interface

The wrapper exposes the standard API:

    reset()
        → observation, info

    step(action)
        → observation
        → reward
        → terminated
        → truncated
        → info

The learning action space is:

$$
a_t \in [-1,1]^3
$$

with shape `(3,)` and dtype `float32`.

The learning observation is the 12-element truth-derived state established in Step 3:

$$
o_t =
[
\Delta x,
\Delta y,
\Delta z,
\phi,
\theta,
\psi,
u,
v,
w,
p,
q,
r
]^T.
$$

The Gymnasium observation space therefore has shape `(12,)` and dtype `float32`.

### Episode Semantics

Waypoint success produces:

    terminated = True
    truncated  = False

Exhausting the maximum episode horizon without success produces:

    terminated = False
    truncated  = True

If success occurs on the last permitted transition, success takes precedence over truncation.

### Reset Semantics

Environment reset:

1. initializes Gymnasium's environment RNG;
2. calls the production Simulator's own `reset()`;
3. restores the configured physical initial state;
4. restores the Simulator sensor RNG sequence;
5. clears the episode step counter;
6. clears cumulative episode return;
7. restores the initial goal distance;
8. rebuilds the initial 12-element observation.

The task itself currently uses a fixed initial condition, so the environment seed does not yet randomize physical state.

The seed is nevertheless handled through the standard Gymnasium API.

### Diagnostic Information

The `info` dictionary provides:

- `goal_distance`;
- `success`;
- `simulation_time`;
- `episode_step`;
- `episode_return`;
- `production_wrench`.

These diagnostics are not part of the policy observation.

### Numerical Validation Rule

Values copied without arithmetic may be compared exactly where appropriate.

Values generated through floating-point multiplication, especially from Gymnasium `float32` actions, are compared using tight numerical tolerances rather than bitwise equality.

For the production-wrench mapping we require:

$$
|\tau_{actual}-\tau_{expected}| < 10^{-6}.
$$

This is many orders of magnitude smaller than any physically meaningful force scale in this experiment.

### Validation Criteria

This step passes if:

1. the wrapper is a valid `gymnasium.Env`;
2. the action space is `Box(-1, 1, (3,), float32)`;
3. the observation space has shape `(12,)` and dtype `float32`;
4. `reset()` returns a valid observation and info dictionary;
5. the initial observation lies inside the declared observation space;
6. one legal Gymnasium action is accepted;
7. the learning action maps numerically to the expected six-dimensional production wrench;
8. rotational moments remain effectively zero;
9. `step()` returns the standard five Gymnasium outputs;
10. observation and reward values remain finite;
11. simulation time advances exactly one production timestep;
12. episode counters update correctly;
13. the ordinary probe transition is neither terminated nor truncated;
14. repeated reset restores the initial task state;
15. no production source, test, dependency, or artifact is modified.

The multi-episode random-policy stress test will begin only after this interface contract passes.

In [7]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

from hydrolab3d.core import Simulator


# ---------------------------------------------------------------------
# 1. Explicit Notebook 11 seed assignments
# ---------------------------------------------------------------------

ENVIRONMENT_SEED = 22011
ACTION_SPACE_SEED = 22012
SIMULATOR_SENSOR_SEED = 22013


# ---------------------------------------------------------------------
# 2. Notebook-local Gymnasium wrapper
# ---------------------------------------------------------------------

class HydroLabLearningEnv(gym.Env):
    """
    Temporary Notebook 11 learning interface for HydroLab-3D.

    Policy observation:
        12D truth-derived task state

    Policy action:
        normalized translational action in [-1, 1]^3

    Production action:
        generalized body-frame wrench
        [Fx, Fy, Fz, 0, 0, 0]
    """

    metadata = {
        "render_modes": [],
    }

    def __init__(
        self,
        *,
        initial_pose,
        initial_body_velocity,
        dt,
        dynamics_kwargs,
        goal_position,
        force_limits,
        success_radius,
        max_steps,
        sensor_config=None,
        sensor_seed=None,
        action_space_seed=None,
    ):
        super().__init__()

        self.goal_position = np.asarray(
            goal_position,
            dtype=float,
        ).copy()

        self.force_limits = np.asarray(
            force_limits,
            dtype=float,
        ).copy()

        self.success_radius = float(
            success_radius
        )

        self.max_steps = int(
            max_steps
        )

        if self.goal_position.shape != (3,):
            raise ValueError(
                "goal_position must have shape (3,)"
            )

        if self.force_limits.shape != (3,):
            raise ValueError(
                "force_limits must have shape (3,)"
            )

        if not np.all(
            np.isfinite(
                self.goal_position
            )
        ):
            raise ValueError(
                "goal_position must contain only finite values"
            )

        if not np.all(
            np.isfinite(
                self.force_limits
            )
        ):
            raise ValueError(
                "force_limits must contain only finite values"
            )

        if not np.all(
            self.force_limits > 0.0
        ):
            raise ValueError(
                "force_limits must be strictly positive"
            )

        if (
            not np.isfinite(
                self.success_radius
            )
            or self.success_radius <= 0.0
        ):
            raise ValueError(
                "success_radius must be finite and positive"
            )

        if self.max_steps <= 0:
            raise ValueError(
                "max_steps must be positive"
            )

        self.simulator = Simulator(
            initial_pose=initial_pose,
            initial_body_velocity=(
                initial_body_velocity
            ),
            dt=dt,
            dynamics_kwargs=(
                dynamics_kwargs
            ),
            sensor_config=(
                {}
                if sensor_config is None
                else sensor_config
            ),
            sensor_seed=sensor_seed,
        )

        self.action_space = spaces.Box(
            low=-1.0,
            high=1.0,
            shape=(3,),
            dtype=np.float32,
        )

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(12,),
            dtype=np.float32,
        )

        if action_space_seed is not None:
            self.action_space.seed(
                action_space_seed
            )

        self._episode_step = 0
        self._episode_return = 0.0
        self._previous_distance = None


    def _observation_from_truth(
        self,
        truth,
    ):
        observation = (
            build_learning_observation(
                truth,
                self.goal_position,
            )
        )

        if not self.observation_space.contains(
            observation
        ):
            raise RuntimeError(
                "constructed observation lies outside "
                "the declared Gymnasium observation space"
            )

        return observation


    def _build_info(
        self,
        *,
        truth,
        distance,
        success,
        wrench,
    ):
        return {
            "goal_distance": float(
                distance
            ),
            "success": bool(
                success
            ),
            "simulation_time": float(
                truth["time"]
            ),
            "episode_step": int(
                self._episode_step
            ),
            "episode_return": float(
                self._episode_return
            ),
            "production_wrench": np.asarray(
                wrench,
                dtype=float,
            ).copy(),
        }


    def reset(
        self,
        *,
        seed=None,
        options=None,
    ):
        super().reset(
            seed=seed
        )

        truth = (
            self.simulator.reset()
        )

        self._episode_step = 0
        self._episode_return = 0.0

        self._previous_distance = (
            learning_goal_distance(
                truth,
                self.goal_position,
            )
        )

        observation = (
            self._observation_from_truth(
                truth
            )
        )

        info = self._build_info(
            truth=truth,
            distance=self._previous_distance,
            success=False,
            wrench=np.zeros(
                6,
                dtype=float,
            ),
        )

        return observation, info


    def step(
        self,
        action,
    ):
        action = np.asarray(
            action,
            dtype=float,
        )

        if action.shape != (3,):
            raise ValueError(
                "Gymnasium learning action must have shape (3,)"
            )

        if not np.all(
            np.isfinite(
                action
            )
        ):
            raise ValueError(
                "Gymnasium learning action must contain only finite values"
            )

        (
            clipped_action,
            production_wrench,
        ) = map_learning_action_to_wrench(
            action,
            self.force_limits,
        )

        result = self.simulator.step(
            production_wrench
        )

        truth = result["truth"]

        observation = (
            self._observation_from_truth(
                truth
            )
        )

        current_distance = (
            learning_goal_distance(
                truth,
                self.goal_position,
            )
        )

        success = bool(
            current_distance
            <= self.success_radius
        )

        reward = (
            calculate_learning_reward(
                self._previous_distance,
                current_distance,
                clipped_action,
                success=success,
            )
        )

        self._episode_step += 1

        self._episode_return += (
            reward
        )

        terminated = bool(
            success
        )

        truncated = bool(
            self._episode_step
            >= self.max_steps
            and not terminated
        )

        self._previous_distance = (
            current_distance
        )

        info = self._build_info(
            truth=truth,
            distance=current_distance,
            success=success,
            wrench=production_wrench,
        )

        return (
            observation,
            float(reward),
            terminated,
            truncated,
            info,
        )


# ---------------------------------------------------------------------
# 3. Construct temporary learning environment
# ---------------------------------------------------------------------

learning_env = HydroLabLearningEnv(
    initial_pose=contract_initial_pose,
    initial_body_velocity=(
        contract_initial_body_velocity
    ),
    dt=contract_dt,
    dynamics_kwargs=(
        contract_dynamics_kwargs
    ),
    goal_position=(
        LEARNING_GOAL_POSITION
    ),
    force_limits=(
        LEARNING_FORCE_LIMITS
    ),
    success_radius=(
        LEARNING_SUCCESS_RADIUS
    ),
    max_steps=(
        LEARNING_MAX_STEPS
    ),
    sensor_config=(
        contract_sensor_config
    ),
    sensor_seed=(
        SIMULATOR_SENSOR_SEED
    ),
    action_space_seed=(
        ACTION_SPACE_SEED
    ),
)


# ---------------------------------------------------------------------
# 4. Validate environment identity and declared spaces
# ---------------------------------------------------------------------

assert isinstance(
    learning_env,
    gym.Env,
)

assert (
    learning_env.action_space.shape
    == (3,)
)

assert (
    learning_env.action_space.dtype
    == np.float32
)

assert np.array_equal(
    learning_env.action_space.low,
    np.full(
        3,
        -1.0,
        dtype=np.float32,
    ),
)

assert np.array_equal(
    learning_env.action_space.high,
    np.full(
        3,
        1.0,
        dtype=np.float32,
    ),
)

assert (
    learning_env.observation_space.shape
    == (12,)
)

assert (
    learning_env.observation_space.dtype
    == np.float32
)


# ---------------------------------------------------------------------
# 5. Validate reset()
# ---------------------------------------------------------------------

(
    env_initial_observation,
    env_initial_info,
) = learning_env.reset(
    seed=ENVIRONMENT_SEED
)

assert (
    env_initial_observation.shape
    == (12,)
)

assert (
    env_initial_observation.dtype
    == np.float32
)

assert learning_env.observation_space.contains(
    env_initial_observation
)

assert np.all(
    np.isfinite(
        env_initial_observation
    )
)

assert (
    env_initial_info[
        "episode_step"
    ]
    == 0
)

assert (
    env_initial_info[
        "simulation_time"
    ]
    == 0.0
)

assert np.isclose(
    env_initial_info[
        "goal_distance"
    ],
    task_initial_distance,
    rtol=0.0,
    atol=1e-12,
)

assert (
    env_initial_info[
        "episode_return"
    ]
    == 0.0
)


# ---------------------------------------------------------------------
# 6. Validate one legal Gymnasium step
# ---------------------------------------------------------------------

env_probe_action = np.array(
    [
        0.50,
        -0.25,
        0.10,
    ],
    dtype=np.float32,
)

assert learning_env.action_space.contains(
    env_probe_action
)

(
    env_next_observation,
    env_reward,
    env_terminated,
    env_truncated,
    env_info,
) = learning_env.step(
    env_probe_action
)


# ---------------------------------------------------------------------
# 7. Validate standard five-value step contract
# ---------------------------------------------------------------------

assert (
    env_next_observation.shape
    == (12,)
)

assert (
    env_next_observation.dtype
    == np.float32
)

assert learning_env.observation_space.contains(
    env_next_observation
)

assert np.all(
    np.isfinite(
        env_next_observation
    )
)

assert np.isfinite(
    env_reward
)

assert isinstance(
    env_reward,
    float,
)

assert isinstance(
    env_terminated,
    bool,
)

assert isinstance(
    env_truncated,
    bool,
)

assert not env_terminated

assert not env_truncated


# ---------------------------------------------------------------------
# 8. Validate production-wrench mapping numerically
#
# Gymnasium action is float32, so mapping must use tolerance-based
# floating-point comparison rather than exact bitwise equality.
# ---------------------------------------------------------------------

expected_env_wrench = np.array(
    [
        10.0,
        -5.0,
        2.0,
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)

actual_env_wrench = np.asarray(
    env_info[
        "production_wrench"
    ],
    dtype=float,
)

wrench_error = (
    actual_env_wrench
    - expected_env_wrench
)

maximum_wrench_error = float(
    np.max(
        np.abs(
            wrench_error
        )
    )
)

assert actual_env_wrench.shape == (6,)

assert np.allclose(
    actual_env_wrench,
    expected_env_wrench,
    rtol=0.0,
    atol=1e-6,
)

assert np.allclose(
    actual_env_wrench[3:],
    np.zeros(
        3,
        dtype=float,
    ),
    rtol=0.0,
    atol=1e-12,
)


# ---------------------------------------------------------------------
# 9. Validate counters and simulator time
# ---------------------------------------------------------------------

assert (
    env_info[
        "episode_step"
    ]
    == 1
)

assert np.isclose(
    env_info[
        "simulation_time"
    ],
    contract_dt,
    rtol=0.0,
    atol=1e-15,
)

assert np.isclose(
    env_info[
        "episode_return"
    ],
    env_reward,
    rtol=0.0,
    atol=1e-15,
)


# ---------------------------------------------------------------------
# 10. Repeated reset must restore exact initial task state
# ---------------------------------------------------------------------

(
    env_reset_observation,
    env_reset_info,
) = learning_env.reset(
    seed=ENVIRONMENT_SEED
)

assert np.array_equal(
    env_reset_observation,
    env_initial_observation,
)

assert np.isclose(
    env_reset_info[
        "goal_distance"
    ],
    env_initial_info[
        "goal_distance"
    ],
    rtol=0.0,
    atol=0.0,
)

assert (
    env_reset_info[
        "episode_step"
    ]
    == 0
)

assert (
    env_reset_info[
        "simulation_time"
    ]
    == 0.0
)

assert (
    env_reset_info[
        "episode_return"
    ]
    == 0.0
)


# ---------------------------------------------------------------------
# 11. Explicit finite-value checks on diagnostics
# ---------------------------------------------------------------------

assert np.isfinite(
    env_info[
        "goal_distance"
    ]
)

assert np.isfinite(
    env_info[
        "simulation_time"
    ]
)

assert np.isfinite(
    env_info[
        "episode_return"
    ]
)

assert np.all(
    np.isfinite(
        actual_env_wrench
    )
)


# ---------------------------------------------------------------------
# 12. Concise report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 11 - STEP 4 GYMNASIUM WRAPPER")
print("=" * 80)


print("\nEnvironment")
print("-" * 80)

print(
    "class                 :",
    type(
        learning_env
    ).__name__,
)

print(
    "Gymnasium Env         :",
    isinstance(
        learning_env,
        gym.Env,
    ),
)

print(
    f"environment seed      : "
    f"{ENVIRONMENT_SEED}"
)

print(
    f"action-space seed     : "
    f"{ACTION_SPACE_SEED}"
)

print(
    f"simulator sensor seed : "
    f"{SIMULATOR_SENSOR_SEED}"
)


print("\nSpaces")
print("-" * 80)

print(
    "action space          :",
    learning_env.action_space,
)

print(
    "observation space     :",
    learning_env.observation_space,
)

print(
    "action shape          :",
    learning_env.action_space.shape,
)

print(
    "observation shape     :",
    learning_env.observation_space.shape,
)


print("\nReset contract")
print("-" * 80)

print(
    "observation           :",
    env_initial_observation,
)

print(
    f"goal distance         : "
    f"{env_initial_info['goal_distance']:.9f} m"
)

print(
    f"simulation time       : "
    f"{env_initial_info['simulation_time']:.6f} s"
)

print(
    "episode step          :",
    env_initial_info[
        "episode_step"
    ],
)

print(
    "observation contained :",
    learning_env.observation_space.contains(
        env_initial_observation
    ),
)


print("\nOne-step contract")
print("-" * 80)

print(
    "learning action       :",
    env_probe_action,
)

print(
    "production wrench     :",
    actual_env_wrench,
)

print(
    "expected wrench       :",
    expected_env_wrench,
)

print(
    "wrench error          :",
    wrench_error,
)

print(
    f"max |wrench error|    : "
    f"{maximum_wrench_error:.12e}"
)

print(
    f"reward                : "
    f"{env_reward:.12f}"
)

print(
    f"goal distance         : "
    f"{env_info['goal_distance']:.9f} m"
)

print(
    f"terminated            : "
    f"{env_terminated}"
)

print(
    f"truncated             : "
    f"{env_truncated}"
)

print(
    f"episode step          : "
    f"{env_info['episode_step']}"
)

print(
    f"simulation time       : "
    f"{env_info['simulation_time']:.6f} s"
)


print("\nRepeated reset")
print("-" * 80)

print(
    "initial observation exact :",
    np.array_equal(
        env_initial_observation,
        env_reset_observation,
    ),
)

print(
    "goal distance exact       :",
    env_initial_info[
        "goal_distance"
    ]
    == env_reset_info[
        "goal_distance"
    ],
)

print(
    "step counter restored     :",
    env_reset_info[
        "episode_step"
    ]
    == 0,
)

print(
    "episode return restored   :",
    env_reset_info[
        "episode_return"
    ]
    == 0.0,
)


print("\n" + "=" * 80)
print("STEP 4 GYMNASIUM WRAPPER PASSED")
print("=" * 80)

print(
    "The temporary Notebook 11 Gymnasium wrapper translates bounded "
    "3D float32 learning actions into the existing HydroLab-3D "
    "six-axis production wrench."
)

print(
    "The observed float32 mapping error is below 1e-6 N and is purely "
    "numerical representation error, not an interface mismatch."
)

print(
    "reset() and step() satisfy the Gymnasium interaction contract, "
    "and repeated reset restores the exact fixed task state."
)

print(
    "The production Simulator remains unchanged and continues to own "
    "physics, deterministic model time, truth state, and sensor state."
)

HYDROLAB-3D NOTEBOOK 11 - STEP 4 GYMNASIUM WRAPPER

Environment
--------------------------------------------------------------------------------
class                 : HydroLabLearningEnv
Gymnasium Env         : True
environment seed      : 22011
action-space seed     : 22012
simulator sensor seed : 22013

Spaces
--------------------------------------------------------------------------------
action space          : Box(-1.0, 1.0, (3,), float32)
observation space     : Box(-inf, inf, (12,), float32)
action shape          : (3,)
observation shape     : (12,)

Reset contract
--------------------------------------------------------------------------------
observation           : [ 6. -2.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
goal distance         : 6.403124237 m
simulation time       : 0.000000 s
episode step          : 0
observation contained : True

One-step contract
--------------------------------------------------------------------------------
learning action       : [ 0.5  -0.25

## Step 5 - Gymnasium Compliance and Random-Policy Stress Validation

The temporary learning environment passed its direct reset/step contract in Step 4.

Before any PPO model is created, the environment must now survive a broader interface and numerical stress test.

This step has two purposes:

### 1. Framework Compatibility

The notebook-local environment will be checked against:

- Gymnasium's environment checker;
- Stable-Baselines3's environment checker.

These checks help detect interface problems such as:

- incorrect return signatures;
- inconsistent observation shapes;
- invalid dtypes;
- incompatible action spaces;
- invalid termination flags;
- observations outside the declared space.

Passing these checks does not prove that the learning task is good.

It only demonstrates that the wrapper behaves like a valid learning environment.

### 2. Random-Policy Stress Test

A bounded random policy will then interact with the real production `Simulator` for multiple complete episodes.

The random policy is intentionally unintelligent.

Its purpose is to exercise the interface over many transitions before training begins.

For every transition, the notebook will validate:

- action shape `(3,)`;
- action dtype compatibility;
- action bounds `[-1,1]`;
- observation shape `(12,)`;
- observation dtype `float32`;
- observation-space containment;
- finite observation values;
- finite scalar reward;
- finite goal distance;
- finite episode return;
- finite production wrench;
- production wrench shape `(6,)`;
- translational force limits;
- exactly zero commanded moments;
- monotonically increasing episode step count;
- simulator-time consistency;
- valid `terminated` and `truncated` flags;
- no simultaneous termination and truncation;
- correct horizon truncation;
- correct success termination.

The stress test will also perform repeated resets and verify that each episode starts from the exact fixed benchmark observation.

### Reproducibility

Random stress actions will use a dedicated NumPy random generator:

`RANDOM_STRESS_SEED = 22014`

This is separate from:

- the Gymnasium environment seed;
- the action-space seed;
- the production sensor seed.

The stress rollout will therefore be reproducible without relying on hidden global NumPy state.

### Defensive Input Checks

The environment will also be probed with:

- a wrong-shaped action;
- an action containing `NaN`;
- an action containing `Inf`.

These inputs must be rejected explicitly.

An intentionally out-of-range but finite three-element action will also be tested.

Because the Step 3 contract deliberately defines clipping, that action should be bounded to `[-1,1]` before being converted into physical force.

Thus an input such as:

    [2.0, -3.0, 0.5]

must never produce forces exceeding the declared:

    ±20 N

translational limits.

### Stress-Test Scale

The test will run:

- 10 episodes;
- at most 750 steps per episode;
- at most 7,500 production simulator transitions.

This is large enough to exercise repeated reset/step behavior while remaining a lightweight CPU-only validation.

### Validation Criteria

This step passes if:

1. Gymnasium's environment checker completes successfully;
2. Stable-Baselines3's environment checker completes successfully;
3. all random-policy observations remain finite and inside the declared space;
4. all rewards remain finite;
5. every production wrench remains finite and bounded;
6. no commanded rotational moment is introduced;
7. simulator time remains consistent with step count and `dt`;
8. every completed episode ends through valid termination or truncation;
9. horizon-limited episodes truncate at exactly the declared maximum step count;
10. successful episodes terminate through the declared goal-radius condition;
11. repeated resets restore the exact initial observation;
12. malformed and non-finite actions are rejected;
13. finite out-of-range actions are clipped safely;
14. no unexpected exception, `NaN`, or `Inf` occurs during random stress.

No learning model is created in this step.

PPO training remains blocked until this stress test passes.

In [8]:
import numpy as np

from gymnasium.utils.env_checker import (
    check_env as gymnasium_check_env,
)

from stable_baselines3.common.env_checker import (
    check_env as sb3_check_env,
)


# ---------------------------------------------------------------------
# 1. Dedicated random-stress configuration
# ---------------------------------------------------------------------

RANDOM_STRESS_SEED = 22014
RANDOM_STRESS_EPISODES = 10

random_stress_rng = np.random.default_rng(
    RANDOM_STRESS_SEED
)


# ---------------------------------------------------------------------
# 2. Run standard framework environment checks
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 11 - STEP 5 RANDOM STRESS VALIDATION")
print("=" * 80)


print("\nFramework compatibility")
print("-" * 80)


# Gymnasium checker
gymnasium_check_env(
    learning_env,
    skip_render_check=True,
)

print(
    "Gymnasium environment checker      : PASS"
)


# Stable-Baselines3 checker
sb3_check_env(
    learning_env,
    warn=True,
)

print(
    "Stable-Baselines3 environment check: PASS"
)


# ---------------------------------------------------------------------
# 3. Preserve canonical initial observation for reset comparison
# ---------------------------------------------------------------------

(
    stress_reference_observation,
    stress_reference_info,
) = learning_env.reset(
    seed=ENVIRONMENT_SEED
)

assert (
    stress_reference_observation.shape
    == (12,)
)

assert (
    stress_reference_observation.dtype
    == np.float32
)

assert learning_env.observation_space.contains(
    stress_reference_observation
)

assert np.all(
    np.isfinite(
        stress_reference_observation
    )
)


# ---------------------------------------------------------------------
# 4. Random-policy stress-test storage
# ---------------------------------------------------------------------

stress_episode_steps = []
stress_episode_returns = []
stress_final_distances = []
stress_min_distances = []
stress_successes = []
stress_terminated_flags = []
stress_truncated_flags = []

stress_total_transitions = 0

stress_global_min_observation = np.full(
    12,
    np.inf,
    dtype=float,
)

stress_global_max_observation = np.full(
    12,
    -np.inf,
    dtype=float,
)

stress_max_abs_wrench = np.zeros(
    6,
    dtype=float,
)

stress_max_abs_reward = 0.0


# ---------------------------------------------------------------------
# 5. Execute multiple complete random-policy episodes
# ---------------------------------------------------------------------

for episode_index in range(
    RANDOM_STRESS_EPISODES
):

    episode_seed = (
        ENVIRONMENT_SEED
        + episode_index
    )

    observation, info = learning_env.reset(
        seed=episode_seed
    )

    # -------------------------------------------------------------
    # Every fixed-task reset must return the same physical state.
    # -------------------------------------------------------------

    assert np.array_equal(
        observation,
        stress_reference_observation,
    )

    assert (
        info["episode_step"]
        == 0
    )

    assert (
        info["simulation_time"]
        == 0.0
    )

    assert (
        info["episode_return"]
        == 0.0
    )

    assert np.isclose(
        info["goal_distance"],
        task_initial_distance,
        rtol=0.0,
        atol=1e-12,
    )

    episode_return = 0.0

    episode_min_distance = float(
        info["goal_distance"]
    )

    episode_done = False

    final_terminated = False
    final_truncated = False

    # -------------------------------------------------------------
    # Random rollout
    # -------------------------------------------------------------

    for step_index in range(
        1,
        LEARNING_MAX_STEPS + 1,
    ):

        action = random_stress_rng.uniform(
            low=-1.0,
            high=1.0,
            size=3,
        ).astype(
            np.float32
        )

        # ---------------------------------------------------------
        # Action contract
        # ---------------------------------------------------------

        assert (
            action.shape
            == (3,)
        )

        assert np.all(
            np.isfinite(
                action
            )
        )

        assert np.all(
            action >= -1.0
        )

        assert np.all(
            action <= 1.0
        )

        assert learning_env.action_space.contains(
            action
        )

        (
            next_observation,
            reward,
            terminated,
            truncated,
            step_info,
        ) = learning_env.step(
            action
        )

        stress_total_transitions += 1

        # ---------------------------------------------------------
        # Observation contract
        # ---------------------------------------------------------

        assert (
            next_observation.shape
            == (12,)
        )

        assert (
            next_observation.dtype
            == np.float32
        )

        assert np.all(
            np.isfinite(
                next_observation
            )
        )

        assert learning_env.observation_space.contains(
            next_observation
        )

        # ---------------------------------------------------------
        # Reward contract
        # ---------------------------------------------------------

        assert isinstance(
            reward,
            float,
        )

        assert np.isfinite(
            reward
        )

        episode_return += reward

        stress_max_abs_reward = max(
            stress_max_abs_reward,
            abs(
                float(
                    reward
                )
            ),
        )

        # ---------------------------------------------------------
        # Termination contract
        # ---------------------------------------------------------

        assert isinstance(
            terminated,
            bool,
        )

        assert isinstance(
            truncated,
            bool,
        )

        assert not (
            terminated
            and truncated
        )

        # ---------------------------------------------------------
        # Diagnostic contract
        # ---------------------------------------------------------

        assert (
            step_info[
                "episode_step"
            ]
            == step_index
        )

        expected_simulation_time = (
            step_index
            * contract_dt
        )

        assert np.isclose(
            step_info[
                "simulation_time"
            ],
            expected_simulation_time,
            rtol=0.0,
            atol=1e-11,
        )

        assert np.isfinite(
            step_info[
                "goal_distance"
            ]
        )

        assert np.isfinite(
            step_info[
                "episode_return"
            ]
        )

        assert np.isclose(
            step_info[
                "episode_return"
            ],
            episode_return,
            rtol=0.0,
            atol=1e-10,
        )

        # ---------------------------------------------------------
        # Production-wrench contract
        # ---------------------------------------------------------

        production_wrench = np.asarray(
            step_info[
                "production_wrench"
            ],
            dtype=float,
        )

        assert (
            production_wrench.shape
            == (6,)
        )

        assert np.all(
            np.isfinite(
                production_wrench
            )
        )

        assert np.all(
            np.abs(
                production_wrench[:3]
            )
            <= (
                LEARNING_FORCE_LIMITS
                + 1e-6
            )
        )

        assert np.allclose(
            production_wrench[3:],
            np.zeros(
                3,
                dtype=float,
            ),
            rtol=0.0,
            atol=1e-12,
        )

        # ---------------------------------------------------------
        # Track numerical extrema
        # ---------------------------------------------------------

        stress_global_min_observation = (
            np.minimum(
                stress_global_min_observation,
                next_observation.astype(
                    float
                ),
            )
        )

        stress_global_max_observation = (
            np.maximum(
                stress_global_max_observation,
                next_observation.astype(
                    float
                ),
            )
        )

        stress_max_abs_wrench = np.maximum(
            stress_max_abs_wrench,
            np.abs(
                production_wrench
            ),
        )

        episode_min_distance = min(
            episode_min_distance,
            float(
                step_info[
                    "goal_distance"
                ]
            ),
        )

        # ---------------------------------------------------------
        # Success semantics
        # ---------------------------------------------------------

        if terminated:

            assert (
                step_info[
                    "success"
                ]
                is True
            )

            assert (
                step_info[
                    "goal_distance"
                ]
                <= (
                    LEARNING_SUCCESS_RADIUS
                    + 1e-12
                )
            )

            final_terminated = True
            final_truncated = False
            episode_done = True
            break

        # ---------------------------------------------------------
        # Horizon semantics
        # ---------------------------------------------------------

        if truncated:

            assert (
                step_index
                == LEARNING_MAX_STEPS
            )

            assert (
                step_info[
                    "success"
                ]
                is False
            )

            final_terminated = False
            final_truncated = True
            episode_done = True
            break

        observation = next_observation

    # -------------------------------------------------------------
    # Every episode must end legally.
    # -------------------------------------------------------------

    assert episode_done

    assert (
        final_terminated
        != final_truncated
    )

    stress_episode_steps.append(
        step_info[
            "episode_step"
        ]
    )

    stress_episode_returns.append(
        episode_return
    )

    stress_final_distances.append(
        step_info[
            "goal_distance"
        ]
    )

    stress_min_distances.append(
        episode_min_distance
    )

    stress_successes.append(
        int(
            final_terminated
        )
    )

    stress_terminated_flags.append(
        final_terminated
    )

    stress_truncated_flags.append(
        final_truncated
    )


# ---------------------------------------------------------------------
# 6. Convert stress summaries to arrays
# ---------------------------------------------------------------------

stress_episode_steps = np.asarray(
    stress_episode_steps,
    dtype=int,
)

stress_episode_returns = np.asarray(
    stress_episode_returns,
    dtype=float,
)

stress_final_distances = np.asarray(
    stress_final_distances,
    dtype=float,
)

stress_min_distances = np.asarray(
    stress_min_distances,
    dtype=float,
)

stress_successes = np.asarray(
    stress_successes,
    dtype=int,
)

stress_terminated_flags = np.asarray(
    stress_terminated_flags,
    dtype=bool,
)

stress_truncated_flags = np.asarray(
    stress_truncated_flags,
    dtype=bool,
)


# ---------------------------------------------------------------------
# 7. Global stress-test numerical checks
# ---------------------------------------------------------------------

for array in (
    stress_episode_returns,
    stress_final_distances,
    stress_min_distances,
    stress_global_min_observation,
    stress_global_max_observation,
    stress_max_abs_wrench,
):
    assert np.all(
        np.isfinite(
            array
        )
    )

assert (
    stress_total_transitions
    <= (
        RANDOM_STRESS_EPISODES
        * LEARNING_MAX_STEPS
    )
)

assert np.all(
    stress_episode_steps
    > 0
)

assert np.all(
    stress_episode_steps
    <= LEARNING_MAX_STEPS
)

assert np.all(
    stress_final_distances
    >= 0.0
)

assert np.all(
    stress_min_distances
    >= 0.0
)

assert np.all(
    stress_min_distances
    <= stress_final_distances
    + 1e-12
)

assert np.all(
    stress_max_abs_wrench[:3]
    <= (
        LEARNING_FORCE_LIMITS
        + 1e-6
    )
)

assert np.allclose(
    stress_max_abs_wrench[3:],
    np.zeros(
        3,
        dtype=float,
    ),
    rtol=0.0,
    atol=1e-12,
)


# ---------------------------------------------------------------------
# 8. Malformed-action rejection tests
# ---------------------------------------------------------------------

learning_env.reset(
    seed=ENVIRONMENT_SEED
)


wrong_shape_rejected = False

try:
    learning_env.step(
        np.zeros(
            2,
            dtype=np.float32,
        )
    )
except ValueError:
    wrong_shape_rejected = True

assert wrong_shape_rejected


learning_env.reset(
    seed=ENVIRONMENT_SEED
)


nan_action_rejected = False

try:
    learning_env.step(
        np.array(
            [
                0.0,
                np.nan,
                0.0,
            ],
            dtype=np.float32,
        )
    )
except ValueError:
    nan_action_rejected = True

assert nan_action_rejected


learning_env.reset(
    seed=ENVIRONMENT_SEED
)


inf_action_rejected = False

try:
    learning_env.step(
        np.array(
            [
                0.0,
                np.inf,
                0.0,
            ],
            dtype=np.float32,
        )
    )
except ValueError:
    inf_action_rejected = True

assert inf_action_rejected


# ---------------------------------------------------------------------
# 9. Explicit finite out-of-range clipping test
# ---------------------------------------------------------------------

learning_env.reset(
    seed=ENVIRONMENT_SEED
)

out_of_range_action = np.array(
    [
        2.0,
        -3.0,
        0.5,
    ],
    dtype=np.float32,
)

(
    _,
    clipped_reward,
    clipped_terminated,
    clipped_truncated,
    clipped_info,
) = learning_env.step(
    out_of_range_action
)

clipped_wrench = np.asarray(
    clipped_info[
        "production_wrench"
    ],
    dtype=float,
)

expected_clipped_wrench = np.array(
    [
        20.0,
        -20.0,
        10.0,
        0.0,
        0.0,
        0.0,
    ],
    dtype=float,
)

assert np.allclose(
    clipped_wrench,
    expected_clipped_wrench,
    rtol=0.0,
    atol=1e-6,
)

assert np.isfinite(
    clipped_reward
)

assert not (
    clipped_terminated
    and clipped_truncated
)


# ---------------------------------------------------------------------
# 10. Final reset after all stress probes
# ---------------------------------------------------------------------

(
    final_reset_observation,
    final_reset_info,
) = learning_env.reset(
    seed=ENVIRONMENT_SEED
)

assert np.array_equal(
    final_reset_observation,
    stress_reference_observation,
)

assert (
    final_reset_info[
        "episode_step"
    ]
    == 0
)

assert (
    final_reset_info[
        "simulation_time"
    ]
    == 0.0
)


# ---------------------------------------------------------------------
# 11. Compact stress-test report
# ---------------------------------------------------------------------

print("\nRandom-policy stress configuration")
print("-" * 80)

print(
    f"stress seed             : "
    f"{RANDOM_STRESS_SEED}"
)

print(
    f"episodes                : "
    f"{RANDOM_STRESS_EPISODES}"
)

print(
    f"maximum steps/episode   : "
    f"{LEARNING_MAX_STEPS}"
)

print(
    f"total simulator steps   : "
    f"{stress_total_transitions}"
)


print("\nEpisode outcomes")
print("-" * 80)

print(
    f"successful episodes     : "
    f"{int(np.sum(stress_successes))}"
)

print(
    f"truncated episodes      : "
    f"{int(np.sum(stress_truncated_flags))}"
)

print(
    f"mean episode length     : "
    f"{np.mean(stress_episode_steps):.3f}"
)

print(
    f"mean episode return     : "
    f"{np.mean(stress_episode_returns):.6f}"
)

print(
    f"mean final distance     : "
    f"{np.mean(stress_final_distances):.6f} m"
)

print(
    f"mean minimum distance   : "
    f"{np.mean(stress_min_distances):.6f} m"
)


print("\nNumerical stress envelope")
print("-" * 80)

print(
    "minimum observation    :",
    stress_global_min_observation,
)

print(
    "maximum observation    :",
    stress_global_max_observation,
)

print(
    "maximum |wrench|       :",
    stress_max_abs_wrench,
)

print(
    f"maximum |reward|        : "
    f"{stress_max_abs_reward:.6f}"
)


print("\nDefensive action handling")
print("-" * 80)

print(
    "wrong shape rejected   :",
    wrong_shape_rejected,
)

print(
    "NaN action rejected    :",
    nan_action_rejected,
)

print(
    "Inf action rejected    :",
    inf_action_rejected,
)

print(
    "out-of-range input     :",
    out_of_range_action,
)

print(
    "clipped wrench         :",
    clipped_wrench,
)


print("\nFinal reset")
print("-" * 80)

print(
    "initial observation exact :",
    np.array_equal(
        final_reset_observation,
        stress_reference_observation,
    ),
)

print(
    "step counter restored     :",
    final_reset_info[
        "episode_step"
    ]
    == 0,
)

print(
    "simulation time restored  :",
    final_reset_info[
        "simulation_time"
    ]
    == 0.0,
)


print("\n" + "=" * 80)
print("STEP 5 RANDOM-POLICY STRESS VALIDATION PASSED")
print("=" * 80)

print(
    "The notebook-local environment passed both Gymnasium and "
    "Stable-Baselines3 interface checks."
)

print(
    "Multiple complete bounded-random episodes executed through the "
    "real production Simulator without NaN, Inf, invalid shapes, "
    "time corruption, or unexpected exceptions."
)

print(
    "Malformed and non-finite actions were rejected, while finite "
    "out-of-range actions were clipped to the declared physical "
    "force limits."
)

print(
    "The learning interface is now stable enough to establish the "
    "random-policy baseline before PPO training."
)

HYDROLAB-3D NOTEBOOK 11 - STEP 5 RANDOM STRESS VALIDATION

Framework compatibility
--------------------------------------------------------------------------------
Gymnasium environment checker      : PASS
Stable-Baselines3 environment check: PASS


/home/agniacolyte/HydroLab-3D/.venv/lib/python3.12/site-packages/gymnasium/utils/env_checker.py:317: UserWarning: WARN: A Box observation space minimum value is -infinity. This is probably too low.
  logger.warn(
/home/agniacolyte/HydroLab-3D/.venv/lib/python3.12/site-packages/gymnasium/utils/env_checker.py:321: UserWarning: WARN: A Box observation space maximum value is infinity. This is probably too high.
  logger.warn(



Random-policy stress configuration
--------------------------------------------------------------------------------
stress seed             : 22014
episodes                : 10
maximum steps/episode   : 750
total simulator steps   : 7500

Episode outcomes
--------------------------------------------------------------------------------
successful episodes     : 0
truncated episodes      : 10
mean episode length     : 750.000
mean episode return     : -7.615735
mean final distance     : 4.918944 m
mean minimum distance   : 4.918944 m

Numerical stress envelope
--------------------------------------------------------------------------------
minimum observation    : [ 3.66785407e+00 -2.21665144e+00  4.31803793e-01 -6.43949956e-04
 -1.93569146e-03 -2.68881861e-02 -3.54459882e-02 -1.59570396e-01
 -1.03696585e-01 -1.00715889e-03 -3.29538714e-03 -5.05695725e-03]
maximum observation    : [ 6.04421473e+00 -9.13694084e-01  1.36494303e+00  4.24287864e-04
  2.01706565e-03  5.71810715e-02  2.597667

## Step 6 - Establish the Pre-Training Random-Policy Baseline

Step 5 proved that the temporary learning environment is structurally valid and numerically stable under bounded random actions.

Before PPO training begins, we now establish the formal pre-training baseline against which the learned policy will later be evaluated.

The random-policy stress test from Step 5 was primarily an interface and robustness test. This step is different: it produces the quantitative baseline metrics that will be retained in notebook memory for the later trained-vs-random comparison.

The benchmark remains unchanged:

- fixed initial UUV state;
- fixed waypoint;
- same 12-element truth-derived observation;
- same normalized three-dimensional action space;
- same ±20 N translational force limits;
- same reward;
- same 0.50 m success radius;
- same 750-step episode horizon;
- same production Simulator.

The random policy samples each action component independently from:

$$
a_i \sim U(-1,1).
$$

A dedicated random generator is used:

`BASELINE_POLICY_SEED = 22015`

The evaluation will run 50 independent random-action episodes.

Because the physical initial condition and goal are fixed, variation between baseline episodes comes from the random action sequence rather than from randomized task geometry.

For every episode we will retain:

- episode return;
- final goal distance;
- minimum goal distance;
- number of steps;
- success flag;
- termination flag;
- truncation flag.

We will also preserve one representative random trajectory in notebook memory for later visualization. No persistent artifact is written yet.

### Primary Baseline Metrics

The baseline summary will include:

- success rate;
- mean and median final goal distance;
- mean minimum goal distance;
- mean episode return;
- mean episode length.

The most important quantities for the later learning comparison are final goal distance and success rate.

### Validation Criteria

This step passes if:

1. all 50 baseline episodes complete legally;
2. every episode ends by either success termination or horizon truncation;
3. no episode is simultaneously terminated and truncated;
4. all stored metrics remain finite;
5. all final and minimum distances are non-negative;
6. minimum distance never exceeds final distance except for numerical tolerance;
7. episode lengths remain within the declared horizon;
8. the baseline success rate is computed explicitly;
9. the complete baseline metric arrays remain available for later comparison;
10. no training occurs and no persistent artifact is written.

After this baseline is frozen, the next stage can create the PPO model and perform the first short learning smoke test.

In [9]:
import numpy as np


# ---------------------------------------------------------------------
# 1. Formal random-baseline configuration
# ---------------------------------------------------------------------

BASELINE_POLICY_SEED = 22015
BASELINE_EPISODES = 50
BASELINE_SENSOR_SEED = 22016

baseline_rng = np.random.default_rng(
    BASELINE_POLICY_SEED
)


# ---------------------------------------------------------------------
# 2. Use a separate environment instance for baseline evaluation
# ---------------------------------------------------------------------

baseline_env = HydroLabLearningEnv(
    initial_pose=contract_initial_pose,
    initial_body_velocity=(
        contract_initial_body_velocity
    ),
    dt=contract_dt,
    dynamics_kwargs=(
        contract_dynamics_kwargs
    ),
    goal_position=(
        LEARNING_GOAL_POSITION
    ),
    force_limits=(
        LEARNING_FORCE_LIMITS
    ),
    success_radius=(
        LEARNING_SUCCESS_RADIUS
    ),
    max_steps=(
        LEARNING_MAX_STEPS
    ),
    sensor_config=(
        contract_sensor_config
    ),
    sensor_seed=(
        BASELINE_SENSOR_SEED
    ),
    action_space_seed=(
        ACTION_SPACE_SEED
    ),
)


# ---------------------------------------------------------------------
# 3. Storage for formal baseline metrics
# ---------------------------------------------------------------------

baseline_episode_returns = np.empty(
    BASELINE_EPISODES,
    dtype=float,
)

baseline_final_distances = np.empty(
    BASELINE_EPISODES,
    dtype=float,
)

baseline_min_distances = np.empty(
    BASELINE_EPISODES,
    dtype=float,
)

baseline_episode_steps = np.empty(
    BASELINE_EPISODES,
    dtype=int,
)

baseline_successes = np.empty(
    BASELINE_EPISODES,
    dtype=bool,
)

baseline_terminated = np.empty(
    BASELINE_EPISODES,
    dtype=bool,
)

baseline_truncated = np.empty(
    BASELINE_EPISODES,
    dtype=bool,
)

baseline_trajectories = []


# ---------------------------------------------------------------------
# 4. Execute random-policy evaluation
# ---------------------------------------------------------------------

for episode_index in range(
    BASELINE_EPISODES
):

    episode_seed = (
        ENVIRONMENT_SEED
        + 1000
        + episode_index
    )

    observation, info = baseline_env.reset(
        seed=episode_seed
    )

    assert np.all(
        np.isfinite(
            observation
        )
    )

    assert baseline_env.observation_space.contains(
        observation
    )

    episode_return = 0.0

    episode_min_distance = float(
        info[
            "goal_distance"
        ]
    )

    positions = [
        baseline_env.simulator.state()[
            "pose"
        ][:3].copy()
    ]

    final_terminated = False
    final_truncated = False

    for step_index in range(
        1,
        LEARNING_MAX_STEPS + 1,
    ):

        action = baseline_rng.uniform(
            low=-1.0,
            high=1.0,
            size=3,
        ).astype(
            np.float32
        )

        (
            observation,
            reward,
            terminated,
            truncated,
            info,
        ) = baseline_env.step(
            action
        )

        # ---------------------------------------------------------
        # Runtime validation
        # ---------------------------------------------------------

        assert np.all(
            np.isfinite(
                observation
            )
        )

        assert baseline_env.observation_space.contains(
            observation
        )

        assert np.isfinite(
            reward
        )

        assert np.isfinite(
            info[
                "goal_distance"
            ]
        )

        assert not (
            terminated
            and truncated
        )

        episode_return += float(
            reward
        )

        episode_min_distance = min(
            episode_min_distance,
            float(
                info[
                    "goal_distance"
                ]
            ),
        )

        positions.append(
            baseline_env.simulator.state()[
                "pose"
            ][:3].copy()
        )

        if terminated or truncated:

            final_terminated = bool(
                terminated
            )

            final_truncated = bool(
                truncated
            )

            break

    # -------------------------------------------------------------
    # Episode must finish through exactly one legal mechanism
    # -------------------------------------------------------------

    assert (
        final_terminated
        != final_truncated
    )

    assert (
        1
        <= info[
            "episode_step"
        ]
        <= LEARNING_MAX_STEPS
    )

    if final_terminated:

        assert info[
            "success"
        ]

        assert (
            info[
                "goal_distance"
            ]
            <= (
                LEARNING_SUCCESS_RADIUS
                + 1e-12
            )
        )

    if final_truncated:

        assert (
            info[
                "episode_step"
            ]
            == LEARNING_MAX_STEPS
        )

        assert not info[
            "success"
        ]

    # -------------------------------------------------------------
    # Store episode metrics
    # -------------------------------------------------------------

    baseline_episode_returns[
        episode_index
    ] = episode_return

    baseline_final_distances[
        episode_index
    ] = float(
        info[
            "goal_distance"
        ]
    )

    baseline_min_distances[
        episode_index
    ] = episode_min_distance

    baseline_episode_steps[
        episode_index
    ] = int(
        info[
            "episode_step"
        ]
    )

    baseline_successes[
        episode_index
    ] = bool(
        info[
            "success"
        ]
    )

    baseline_terminated[
        episode_index
    ] = final_terminated

    baseline_truncated[
        episode_index
    ] = final_truncated

    baseline_trajectories.append(
        np.asarray(
            positions,
            dtype=float,
        )
    )


# ---------------------------------------------------------------------
# 5. Global baseline validation
# ---------------------------------------------------------------------

for array in (
    baseline_episode_returns,
    baseline_final_distances,
    baseline_min_distances,
):
    assert np.all(
        np.isfinite(
            array
        )
    )

assert np.all(
    baseline_final_distances
    >= 0.0
)

assert np.all(
    baseline_min_distances
    >= 0.0
)

assert np.all(
    baseline_min_distances
    <= (
        baseline_final_distances
        + 1e-12
    )
)

assert np.all(
    baseline_episode_steps
    >= 1
)

assert np.all(
    baseline_episode_steps
    <= LEARNING_MAX_STEPS
)

assert np.all(
    baseline_terminated
    != baseline_truncated
)


# ---------------------------------------------------------------------
# 6. Compute formal baseline summary
# ---------------------------------------------------------------------

baseline_success_rate = float(
    np.mean(
        baseline_successes
    )
)

baseline_mean_return = float(
    np.mean(
        baseline_episode_returns
    )
)

baseline_median_return = float(
    np.median(
        baseline_episode_returns
    )
)

baseline_mean_final_distance = float(
    np.mean(
        baseline_final_distances
    )
)

baseline_median_final_distance = float(
    np.median(
        baseline_final_distances
    )
)

baseline_mean_min_distance = float(
    np.mean(
        baseline_min_distances
    )
)

baseline_median_min_distance = float(
    np.median(
        baseline_min_distances
    )
)

baseline_mean_steps = float(
    np.mean(
        baseline_episode_steps
    )
)


# ---------------------------------------------------------------------
# 7. Select one representative random trajectory
#
# Choose the episode whose final distance is closest to the median
# final distance. This avoids cherry-picking the best or worst run.
# ---------------------------------------------------------------------

baseline_representative_index = int(
    np.argmin(
        np.abs(
            baseline_final_distances
            - baseline_median_final_distance
        )
    )
)

baseline_representative_trajectory = (
    baseline_trajectories[
        baseline_representative_index
    ].copy()
)

baseline_representative_final_distance = float(
    baseline_final_distances[
        baseline_representative_index
    ]
)


# ---------------------------------------------------------------------
# 8. Keep compact summary in notebook memory for later comparison
# ---------------------------------------------------------------------

baseline_summary = {
    "episodes": BASELINE_EPISODES,
    "policy_seed": BASELINE_POLICY_SEED,
    "success_rate": baseline_success_rate,
    "mean_return": baseline_mean_return,
    "median_return": baseline_median_return,
    "mean_final_distance": (
        baseline_mean_final_distance
    ),
    "median_final_distance": (
        baseline_median_final_distance
    ),
    "mean_min_distance": (
        baseline_mean_min_distance
    ),
    "median_min_distance": (
        baseline_median_min_distance
    ),
    "mean_steps": baseline_mean_steps,
}


# ---------------------------------------------------------------------
# 9. Report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 11 - STEP 6 RANDOM BASELINE")
print("=" * 80)


print("\nEvaluation configuration")
print("-" * 80)

print(
    f"episodes                 : "
    f"{BASELINE_EPISODES}"
)

print(
    f"random-policy seed       : "
    f"{BASELINE_POLICY_SEED}"
)

print(
    f"sensor seed              : "
    f"{BASELINE_SENSOR_SEED}"
)

print(
    f"maximum steps/episode    : "
    f"{LEARNING_MAX_STEPS}"
)


print("\nBaseline performance")
print("-" * 80)

print(
    f"successes                : "
    f"{int(np.sum(baseline_successes))}"
    f"/{BASELINE_EPISODES}"
)

print(
    f"success rate             : "
    f"{baseline_success_rate:.3f}"
)

print(
    f"mean final distance      : "
    f"{baseline_mean_final_distance:.6f} m"
)

print(
    f"median final distance    : "
    f"{baseline_median_final_distance:.6f} m"
)

print(
    f"mean minimum distance    : "
    f"{baseline_mean_min_distance:.6f} m"
)

print(
    f"median minimum distance  : "
    f"{baseline_median_min_distance:.6f} m"
)

print(
    f"mean episode return      : "
    f"{baseline_mean_return:.6f}"
)

print(
    f"median episode return    : "
    f"{baseline_median_return:.6f}"
)

print(
    f"mean episode length      : "
    f"{baseline_mean_steps:.3f} steps"
)


print("\nDistance range")
print("-" * 80)

print(
    f"best final distance      : "
    f"{np.min(baseline_final_distances):.6f} m"
)

print(
    f"worst final distance     : "
    f"{np.max(baseline_final_distances):.6f} m"
)

print(
    f"best minimum distance    : "
    f"{np.min(baseline_min_distances):.6f} m"
)


print("\nRepresentative trajectory")
print("-" * 80)

print(
    f"episode index            : "
    f"{baseline_representative_index}"
)

print(
    f"trajectory samples       : "
    f"{len(baseline_representative_trajectory)}"
)

print(
    f"representative final dist: "
    f"{baseline_representative_final_distance:.6f} m"
)


print("\n" + "=" * 80)
print("STEP 6 RANDOM-POLICY BASELINE PASSED")
print("=" * 80)

print(
    "The formal pre-training random-policy baseline is now frozen "
    "in notebook memory for later comparison with the trained policy."
)

print(
    "No PPO model has been created and no persistent Notebook 11 "
    "artifact has been written yet."
)

HYDROLAB-3D NOTEBOOK 11 - STEP 6 RANDOM BASELINE

Evaluation configuration
--------------------------------------------------------------------------------
episodes                 : 50
random-policy seed       : 22015
sensor seed              : 22016
maximum steps/episode    : 750

Baseline performance
--------------------------------------------------------------------------------
successes                : 0/50
success rate             : 0.000
mean final distance      : 4.856713 m
median final distance    : 4.859576 m
mean minimum distance    : 4.856631 m
median minimum distance  : 4.859576 m
mean episode return      : -7.269693
median episode return    : -7.331489
mean episode length      : 750.000 steps

Distance range
--------------------------------------------------------------------------------
best final distance      : 4.039199 m
worst final distance     : 6.014459 m
best minimum distance    : 4.039199 m

Representative trajectory
--------------------------------------------

## Step 7 - Initialize PPO and Measure the Untrained Policy

The random-policy baseline is now frozen.

Before performing any optimization, we will construct the actual Stable-Baselines3 PPO model and evaluate its **untrained deterministic policy**.

This creates a second pre-training reference:

    random policy
        → purely random bounded actions

    untrained PPO policy
        → neural policy after initialization, before gradient updates

The distinction matters because an initialized neural policy may already produce structured actions even though it has learned nothing.

This step therefore validates the complete connection:

    Stable-Baselines3 PPO
            ↓
    Notebook-local Gymnasium environment
            ↓
    normalized 3D action
            ↓
    production HydroLab-3D Simulator

without performing training yet.

### PPO Configuration

The smoke test will use:

- `MlpPolicy`;
- CPU execution;
- explicit learning seed;
- a small two-layer neural network;
- standard PPO continuous-action behavior;
- no vectorized multi-environment complexity beyond Stable-Baselines3's internal wrapping.

The learning seed is:

`LEARNING_SEED = 22017`

A compact network is sufficient because this notebook is testing simulator compatibility rather than neural-network architecture.

The policy network will use two hidden layers of 64 units each.

### Why Evaluate Before Training?

The untrained policy provides evidence for three things:

1. Stable-Baselines3 can construct a PPO policy against the environment;
2. the neural policy can consume the 12-element observation and produce a legal three-element continuous action;
3. later improvement can be attributed to optimization rather than merely to policy initialization.

### Evaluation Protocol

The untrained PPO policy will be evaluated deterministically on the fixed benchmark task.

Because:

- the physical initial condition is fixed;
- the waypoint is fixed;
- deterministic prediction is used;

repeated episodes may be exactly identical.

If so, they will not be treated as statistically independent trials.

Five repeated rollouts are still executed to verify deterministic replay and interface stability.

For each rollout we record:

- final goal distance;
- minimum goal distance;
- episode return;
- episode length;
- success;
- full XYZ trajectory.

### Validation Criteria

This step passes if:

1. PPO initializes successfully on the notebook-local environment;
2. the policy observation dimension is 12;
3. the policy action dimension is 3;
4. `model.predict()` returns a finite action of shape `(3,)`;
5. the predicted action lies inside the declared action space;
6. complete untrained rollouts execute without exceptions;
7. all observations, rewards, actions, and metrics remain finite;
8. every episode ends through valid termination or truncation;
9. untrained-policy metrics are retained for later comparison;
10. no gradient update or PPO training occurs yet;
11. no production HydroLab-3D source file is modified;
12. no persistent artifact is written.

Once this compatibility check passes, the next step can perform the first actual PPO learning run.

In [10]:
import numpy as np
import torch

from stable_baselines3 import PPO


# ---------------------------------------------------------------------
# 1. Explicit PPO smoke-test configuration
# ---------------------------------------------------------------------

LEARNING_SEED = 22017
UNTRAINED_EVALUATION_EPISODES = 5

PPO_POLICY_KWARGS = {
    "net_arch": [
        64,
        64,
    ],
}


# ---------------------------------------------------------------------
# 2. Separate environment instance for PPO
# ---------------------------------------------------------------------

ppo_env = HydroLabLearningEnv(
    initial_pose=contract_initial_pose,
    initial_body_velocity=(
        contract_initial_body_velocity
    ),
    dt=contract_dt,
    dynamics_kwargs=(
        contract_dynamics_kwargs
    ),
    goal_position=(
        LEARNING_GOAL_POSITION
    ),
    force_limits=(
        LEARNING_FORCE_LIMITS
    ),
    success_radius=(
        LEARNING_SUCCESS_RADIUS
    ),
    max_steps=(
        LEARNING_MAX_STEPS
    ),
    sensor_config=(
        contract_sensor_config
    ),
    sensor_seed=(
        SIMULATOR_SENSOR_SEED
    ),
    action_space_seed=(
        ACTION_SPACE_SEED
    ),
)


# ---------------------------------------------------------------------
# 3. Seed Torch explicitly before model construction
# ---------------------------------------------------------------------

torch.manual_seed(
    LEARNING_SEED
)


# ---------------------------------------------------------------------
# 4. Construct PPO model
#
# No call to learn() occurs in this step.
# ---------------------------------------------------------------------

ppo_model = PPO(
    policy="MlpPolicy",
    env=ppo_env,
    learning_rate=3e-4,
    n_steps=256,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=(
        PPO_POLICY_KWARGS
    ),
    seed=LEARNING_SEED,
    device="cpu",
    verbose=0,
)


# ---------------------------------------------------------------------
# 5. Inspect one deterministic untrained prediction
# ---------------------------------------------------------------------

(
    ppo_probe_observation,
    ppo_probe_info,
) = ppo_env.reset(
    seed=ENVIRONMENT_SEED
)

(
    ppo_probe_action,
    _
) = ppo_model.predict(
    ppo_probe_observation,
    deterministic=True,
)

ppo_probe_action = np.asarray(
    ppo_probe_action,
    dtype=np.float32,
)


assert (
    ppo_probe_observation.shape
    == (12,)
)

assert (
    ppo_probe_action.shape
    == (3,)
)

assert np.all(
    np.isfinite(
        ppo_probe_action
    )
)

assert ppo_env.action_space.contains(
    ppo_probe_action
)


# ---------------------------------------------------------------------
# 6. Untrained-policy rollout helper
# ---------------------------------------------------------------------

def evaluate_deterministic_policy(
    model,
    env,
    number_of_episodes,
):
    episode_returns = []
    final_distances = []
    minimum_distances = []
    episode_steps = []
    successes = []
    terminated_flags = []
    truncated_flags = []
    trajectories = []
    action_histories = []

    for episode_index in range(
        number_of_episodes
    ):

        observation, info = env.reset(
            seed=(
                ENVIRONMENT_SEED
                + 2000
                + episode_index
            )
        )

        episode_return = 0.0

        minimum_distance = float(
            info[
                "goal_distance"
            ]
        )

        trajectory = [
            env.simulator.state()[
                "pose"
            ][:3].copy()
        ]

        actions = []

        final_terminated = False
        final_truncated = False

        for step_index in range(
            1,
            LEARNING_MAX_STEPS + 1,
        ):

            action, _ = model.predict(
                observation,
                deterministic=True,
            )

            action = np.asarray(
                action,
                dtype=np.float32,
            )

            assert (
                action.shape
                == (3,)
            )

            assert np.all(
                np.isfinite(
                    action
                )
            )

            assert env.action_space.contains(
                action
            )

            (
                observation,
                reward,
                terminated,
                truncated,
                info,
            ) = env.step(
                action
            )

            assert np.all(
                np.isfinite(
                    observation
                )
            )

            assert np.isfinite(
                reward
            )

            assert np.isfinite(
                info[
                    "goal_distance"
                ]
            )

            assert not (
                terminated
                and truncated
            )

            episode_return += float(
                reward
            )

            minimum_distance = min(
                minimum_distance,
                float(
                    info[
                        "goal_distance"
                    ]
                ),
            )

            trajectory.append(
                env.simulator.state()[
                    "pose"
                ][:3].copy()
            )

            actions.append(
                action.copy()
            )

            if terminated or truncated:

                final_terminated = bool(
                    terminated
                )

                final_truncated = bool(
                    truncated
                )

                break

        assert (
            final_terminated
            != final_truncated
        )

        episode_returns.append(
            episode_return
        )

        final_distances.append(
            float(
                info[
                    "goal_distance"
                ]
            )
        )

        minimum_distances.append(
            minimum_distance
        )

        episode_steps.append(
            int(
                info[
                    "episode_step"
                ]
            )
        )

        successes.append(
            bool(
                info[
                    "success"
                ]
            )
        )

        terminated_flags.append(
            final_terminated
        )

        truncated_flags.append(
            final_truncated
        )

        trajectories.append(
            np.asarray(
                trajectory,
                dtype=float,
            )
        )

        action_histories.append(
            np.asarray(
                actions,
                dtype=float,
            )
        )

    return {
        "returns": np.asarray(
            episode_returns,
            dtype=float,
        ),
        "final_distances": np.asarray(
            final_distances,
            dtype=float,
        ),
        "minimum_distances": np.asarray(
            minimum_distances,
            dtype=float,
        ),
        "steps": np.asarray(
            episode_steps,
            dtype=int,
        ),
        "successes": np.asarray(
            successes,
            dtype=bool,
        ),
        "terminated": np.asarray(
            terminated_flags,
            dtype=bool,
        ),
        "truncated": np.asarray(
            truncated_flags,
            dtype=bool,
        ),
        "trajectories": trajectories,
        "actions": action_histories,
    }


# ---------------------------------------------------------------------
# 7. Evaluate PPO before any training
# ---------------------------------------------------------------------

untrained_results = (
    evaluate_deterministic_policy(
        ppo_model,
        ppo_env,
        UNTRAINED_EVALUATION_EPISODES,
    )
)


# ---------------------------------------------------------------------
# 8. Validate returned metrics
# ---------------------------------------------------------------------

for array in (
    untrained_results[
        "returns"
    ],
    untrained_results[
        "final_distances"
    ],
    untrained_results[
        "minimum_distances"
    ],
):
    assert np.all(
        np.isfinite(
            array
        )
    )


assert np.all(
    untrained_results[
        "steps"
    ]
    >= 1
)

assert np.all(
    untrained_results[
        "steps"
    ]
    <= LEARNING_MAX_STEPS
)

assert np.all(
    untrained_results[
        "minimum_distances"
    ]
    <= (
        untrained_results[
            "final_distances"
        ]
        + 1e-12
    )
)

assert np.all(
    untrained_results[
        "terminated"
    ]
    != untrained_results[
        "truncated"
    ]
)


# ---------------------------------------------------------------------
# 9. Check deterministic replay behavior
# ---------------------------------------------------------------------

untrained_trajectory_exact = all(
    np.array_equal(
        untrained_results[
            "trajectories"
        ][0],
        trajectory,
    )
    for trajectory in untrained_results[
        "trajectories"
    ][1:]
)

untrained_actions_exact = all(
    np.array_equal(
        untrained_results[
            "actions"
        ][0],
        actions,
    )
    for actions in untrained_results[
        "actions"
    ][1:]
)


# ---------------------------------------------------------------------
# 10. Compact untrained summary
# ---------------------------------------------------------------------

untrained_success_rate = float(
    np.mean(
        untrained_results[
            "successes"
        ]
    )
)

untrained_mean_final_distance = float(
    np.mean(
        untrained_results[
            "final_distances"
        ]
    )
)

untrained_mean_min_distance = float(
    np.mean(
        untrained_results[
            "minimum_distances"
        ]
    )
)

untrained_mean_return = float(
    np.mean(
        untrained_results[
            "returns"
        ]
    )
)

untrained_mean_steps = float(
    np.mean(
        untrained_results[
            "steps"
        ]
    )
)


untrained_summary = {
    "episodes": (
        UNTRAINED_EVALUATION_EPISODES
    ),
    "learning_seed": (
        LEARNING_SEED
    ),
    "success_rate": (
        untrained_success_rate
    ),
    "mean_final_distance": (
        untrained_mean_final_distance
    ),
    "mean_min_distance": (
        untrained_mean_min_distance
    ),
    "mean_return": (
        untrained_mean_return
    ),
    "mean_steps": (
        untrained_mean_steps
    ),
    "deterministic_trajectory_exact": (
        untrained_trajectory_exact
    ),
    "deterministic_actions_exact": (
        untrained_actions_exact
    ),
}


untrained_representative_trajectory = (
    untrained_results[
        "trajectories"
    ][0].copy()
)


# ---------------------------------------------------------------------
# 11. Verify that no training has occurred
# ---------------------------------------------------------------------

assert (
    ppo_model.num_timesteps
    == 0
)


# ---------------------------------------------------------------------
# 12. Report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 11 - STEP 7 UNTRAINED PPO")
print("=" * 80)


print("\nPPO configuration")
print("-" * 80)

print(
    f"learning seed          : "
    f"{LEARNING_SEED}"
)

print(
    "device                 :",
    ppo_model.device,
)

print(
    "policy                 :",
    type(
        ppo_model.policy
    ).__name__,
)

print(
    "network architecture   :",
    PPO_POLICY_KWARGS[
        "net_arch"
    ],
)

print(
    f"PPO n_steps            : "
    f"{ppo_model.n_steps}"
)

print(
    f"PPO batch_size         : "
    f"{ppo_model.batch_size}"
)

print(
    f"timesteps trained      : "
    f"{ppo_model.num_timesteps}"
)


print("\nUntrained policy probe")
print("-" * 80)

print(
    "observation shape      :",
    ppo_probe_observation.shape,
)

print(
    "predicted action       :",
    ppo_probe_action,
)

print(
    "action shape           :",
    ppo_probe_action.shape,
)

print(
    "action inside space    :",
    ppo_env.action_space.contains(
        ppo_probe_action
    ),
)


print("\nUntrained evaluation")
print("-" * 80)

print(
    f"episodes               : "
    f"{UNTRAINED_EVALUATION_EPISODES}"
)

print(
    f"successes              : "
    f"{int(np.sum(untrained_results['successes']))}"
    f"/{UNTRAINED_EVALUATION_EPISODES}"
)

print(
    f"success rate           : "
    f"{untrained_success_rate:.3f}"
)

print(
    f"mean final distance    : "
    f"{untrained_mean_final_distance:.6f} m"
)

print(
    f"mean minimum distance  : "
    f"{untrained_mean_min_distance:.6f} m"
)

print(
    f"mean episode return    : "
    f"{untrained_mean_return:.6f}"
)

print(
    f"mean episode length    : "
    f"{untrained_mean_steps:.3f} steps"
)


print("\nDeterministic replay")
print("-" * 80)

print(
    "trajectory exact across runs :",
    untrained_trajectory_exact,
)

print(
    "actions exact across runs    :",
    untrained_actions_exact,
)


print("\nPre-training references")
print("-" * 80)

print(
    f"random baseline mean final distance : "
    f"{baseline_mean_final_distance:.6f} m"
)

print(
    f"untrained PPO mean final distance   : "
    f"{untrained_mean_final_distance:.6f} m"
)

print(
    f"random baseline success rate        : "
    f"{baseline_success_rate:.3f}"
)

print(
    f"untrained PPO success rate          : "
    f"{untrained_success_rate:.3f}"
)


print("\n" + "=" * 80)
print("STEP 7 UNTRAINED PPO VALIDATION PASSED")
print("=" * 80)

print(
    "Stable-Baselines3 PPO successfully consumes the Notebook 11 "
    "truth-derived observation and produces legal continuous actions "
    "that execute through the real HydroLab-3D production Simulator."
)

print(
    "The model has completed zero training timesteps, so these metrics "
    "represent initialized neural-policy behavior rather than learning."
)

print(
    "Random-policy and untrained-PPO references are now available for "
    "comparison with the first actual PPO training run."
)

HYDROLAB-3D NOTEBOOK 11 - STEP 7 UNTRAINED PPO

PPO configuration
--------------------------------------------------------------------------------
learning seed          : 22017
device                 : cpu
policy                 : ActorCriticPolicy
network architecture   : [64, 64]
PPO n_steps            : 256
PPO batch_size         : 64
timesteps trained      : 0

Untrained policy probe
--------------------------------------------------------------------------------
observation shape      : (12,)
predicted action       : [ 0.00217546 -0.00824257 -0.0067293 ]
action shape           : (3,)
action inside space    : True

Untrained evaluation
--------------------------------------------------------------------------------
episodes               : 5
successes              : 0/5
success rate           : 0.000
mean final distance    : 4.861101 m
mean minimum distance  : 4.861101 m
mean episode return    : 0.209299
mean episode length    : 750.000 steps

Deterministic replay
----------------

## Step 8 - First PPO Learning Smoke Test

The complete learning interface has now passed:

- production Simulator validation;
- task-contract validation;
- Gymnasium validation;
- Gymnasium and Stable-Baselines3 environment checkers;
- 7,500-step bounded-random stress testing;
- a formal 50-episode random baseline;
- an untrained PPO compatibility evaluation.

We can therefore perform the first actual optimization run.

This remains a **learning smoke test**, not an RL performance study.

The objective is simply to determine whether PPO can improve task behavior while interacting with the real production HydroLab-3D Simulator through the notebook-local wrapper.

### Training Budget

The first run will use:

`20,000 environment timesteps`

with the PPO model created in Step 7.

This is intentionally modest.

With the current 750-step episode horizon, 20,000 environment transitions correspond to roughly 26 full-horizon episodes if the policy never succeeds early.

The purpose is not to maximize performance.

The purpose is to ask:

> Does optimization move the initialized policy in a measurably useful direction?

No hyperparameter sweep is performed.

The PPO configuration remains exactly the configuration established in Step 7:

- `MlpPolicy`;
- hidden layers `[64, 64]`;
- learning rate `3e-4`;
- `n_steps = 256`;
- batch size `64`;
- 10 optimization epochs;
- `gamma = 0.99`;
- `gae_lambda = 0.95`;
- `clip_range = 0.2`;
- CPU execution;
- learning seed `22017`.

### Evaluation

Immediately after training, the deterministic learned policy will be evaluated on the same fixed benchmark used before training.

The evaluation contract is unchanged:

- same initial physical state;
- same goal;
- same force limits;
- same reward;
- same success radius;
- same 750-step horizon.

Five deterministic evaluation rollouts will be executed.

Because the benchmark is deterministic, identical learned trajectories across repeated evaluation runs are expected and will be reported honestly rather than interpreted as independent statistical samples.

### Comparison

The trained policy will be compared against both pre-training references:

1. the 50-episode random-policy baseline;
2. the deterministic untrained PPO policy.

The principal physical metrics are:

- final goal distance;
- minimum goal distance;
- success rate.

Episode return is retained as a secondary task metric.

A useful first learning result does not require success.

For example, a substantial reduction in final goal distance compared with both random and untrained behavior would already constitute evidence that optimization is affecting the policy in the intended direction.

Success within the 0.50 m goal region would provide stronger evidence.

### Validation Criteria

This step passes as an execution step if:

1. PPO completes the requested optimization timesteps without numerical failure;
2. the production Simulator remains finite during training;
3. the trained policy continues to produce legal actions;
4. deterministic post-training evaluation completes normally;
5. all post-training metrics remain finite;
6. the trained result is compared numerically with both pre-training references.

A lack of improvement will not be hidden or relabeled as success.

If this first modest training run is insufficient, the next step will diagnose the evidence before changing training effort or task design.

In [11]:
import time
import numpy as np


# ---------------------------------------------------------------------
# 1. First PPO smoke-training budget
# ---------------------------------------------------------------------

PPO_FIRST_TRAINING_TIMESTEPS = 20_000
TRAINED_EVALUATION_EPISODES = 5


# ---------------------------------------------------------------------
# 2. Confirm model is still genuinely untrained before learn()
# ---------------------------------------------------------------------

assert (
    ppo_model.num_timesteps
    == 0
), (
    "PPO model already contains training timesteps. "
    "Step 8 expects the untrained Step 7 model."
)


# ---------------------------------------------------------------------
# 3. Train PPO through the notebook-local environment
#
# The environment internally calls the real production Simulator.
# ---------------------------------------------------------------------

training_start_time = time.perf_counter()

ppo_model.learn(
    total_timesteps=(
        PPO_FIRST_TRAINING_TIMESTEPS
    ),
    reset_num_timesteps=True,
    progress_bar=False,
)

training_elapsed_seconds = (
    time.perf_counter()
    - training_start_time
)


# ---------------------------------------------------------------------
# 4. Validate completed optimization count
#
# Stable-Baselines3 may slightly exceed total_timesteps because PPO
# collects complete rollout buffers of n_steps transitions.
# ---------------------------------------------------------------------

assert (
    ppo_model.num_timesteps
    >= PPO_FIRST_TRAINING_TIMESTEPS
)

assert np.isfinite(
    training_elapsed_seconds
)

assert (
    training_elapsed_seconds
    > 0.0
)


# ---------------------------------------------------------------------
# 5. Probe trained deterministic action from initial observation
# ---------------------------------------------------------------------

(
    trained_probe_observation,
    trained_probe_info,
) = ppo_env.reset(
    seed=ENVIRONMENT_SEED
)

(
    trained_probe_action,
    _
) = ppo_model.predict(
    trained_probe_observation,
    deterministic=True,
)

trained_probe_action = np.asarray(
    trained_probe_action,
    dtype=np.float32,
)


assert (
    trained_probe_action.shape
    == (3,)
)

assert np.all(
    np.isfinite(
        trained_probe_action
    )
)

assert ppo_env.action_space.contains(
    trained_probe_action
)


# ---------------------------------------------------------------------
# 6. Evaluate learned deterministic policy using the same helper and
#    the same fixed benchmark used before training.
# ---------------------------------------------------------------------

trained_results = (
    evaluate_deterministic_policy(
        ppo_model,
        ppo_env,
        TRAINED_EVALUATION_EPISODES,
    )
)


# ---------------------------------------------------------------------
# 7. Validate post-training metrics
# ---------------------------------------------------------------------

for array in (
    trained_results[
        "returns"
    ],
    trained_results[
        "final_distances"
    ],
    trained_results[
        "minimum_distances"
    ],
):
    assert np.all(
        np.isfinite(
            array
        )
    )


assert np.all(
    trained_results[
        "steps"
    ]
    >= 1
)

assert np.all(
    trained_results[
        "steps"
    ]
    <= LEARNING_MAX_STEPS
)

assert np.all(
    trained_results[
        "minimum_distances"
    ]
    <= (
        trained_results[
            "final_distances"
        ]
        + 1e-12
    )
)

assert np.all(
    trained_results[
        "terminated"
    ]
    != trained_results[
        "truncated"
    ]
)


# ---------------------------------------------------------------------
# 8. Deterministic replay check after training
# ---------------------------------------------------------------------

trained_trajectory_exact = all(
    np.array_equal(
        trained_results[
            "trajectories"
        ][0],
        trajectory,
    )
    for trajectory in trained_results[
        "trajectories"
    ][1:]
)

trained_actions_exact = all(
    np.array_equal(
        trained_results[
            "actions"
        ][0],
        actions,
    )
    for actions in trained_results[
        "actions"
    ][1:]
)


# ---------------------------------------------------------------------
# 9. Compute trained-policy summary
# ---------------------------------------------------------------------

trained_success_rate = float(
    np.mean(
        trained_results[
            "successes"
        ]
    )
)

trained_mean_final_distance = float(
    np.mean(
        trained_results[
            "final_distances"
        ]
    )
)

trained_mean_min_distance = float(
    np.mean(
        trained_results[
            "minimum_distances"
        ]
    )
)

trained_mean_return = float(
    np.mean(
        trained_results[
            "returns"
        ]
    )
)

trained_mean_steps = float(
    np.mean(
        trained_results[
            "steps"
        ]
    )
)


# ---------------------------------------------------------------------
# 10. Physical improvement calculations
# ---------------------------------------------------------------------

improvement_vs_random_distance = (
    baseline_mean_final_distance
    - trained_mean_final_distance
)

improvement_vs_untrained_distance = (
    untrained_mean_final_distance
    - trained_mean_final_distance
)

relative_improvement_vs_random = (
    improvement_vs_random_distance
    / baseline_mean_final_distance
)

relative_improvement_vs_untrained = (
    improvement_vs_untrained_distance
    / untrained_mean_final_distance
)


return_improvement_vs_random = (
    trained_mean_return
    - baseline_mean_return
)

return_improvement_vs_untrained = (
    trained_mean_return
    - untrained_mean_return
)


# ---------------------------------------------------------------------
# 11. Determine evidence status without forcing a positive conclusion
# ---------------------------------------------------------------------

distance_better_than_random = bool(
    trained_mean_final_distance
    < baseline_mean_final_distance
)

distance_better_than_untrained = bool(
    trained_mean_final_distance
    < untrained_mean_final_distance
)

success_better_than_random = bool(
    trained_success_rate
    > baseline_success_rate
)

success_better_than_untrained = bool(
    trained_success_rate
    > untrained_success_rate
)

first_training_improved = bool(
    (
        distance_better_than_random
        and distance_better_than_untrained
    )
    or (
        success_better_than_random
        and success_better_than_untrained
    )
)


# ---------------------------------------------------------------------
# 12. Preserve learned trajectory and summary in notebook memory
# ---------------------------------------------------------------------

trained_representative_trajectory = (
    trained_results[
        "trajectories"
    ][0].copy()
)

trained_summary = {
    "requested_training_timesteps": (
        PPO_FIRST_TRAINING_TIMESTEPS
    ),
    "actual_training_timesteps": int(
        ppo_model.num_timesteps
    ),
    "training_elapsed_seconds": float(
        training_elapsed_seconds
    ),
    "evaluation_episodes": (
        TRAINED_EVALUATION_EPISODES
    ),
    "success_rate": (
        trained_success_rate
    ),
    "mean_final_distance": (
        trained_mean_final_distance
    ),
    "mean_min_distance": (
        trained_mean_min_distance
    ),
    "mean_return": (
        trained_mean_return
    ),
    "mean_steps": (
        trained_mean_steps
    ),
    "distance_improvement_vs_random": (
        improvement_vs_random_distance
    ),
    "distance_improvement_vs_untrained": (
        improvement_vs_untrained_distance
    ),
    "relative_improvement_vs_random": (
        relative_improvement_vs_random
    ),
    "relative_improvement_vs_untrained": (
        relative_improvement_vs_untrained
    ),
    "deterministic_trajectory_exact": (
        trained_trajectory_exact
    ),
    "deterministic_actions_exact": (
        trained_actions_exact
    ),
    "first_training_improved": (
        first_training_improved
    ),
}


# ---------------------------------------------------------------------
# 13. Report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 11 - STEP 8 FIRST PPO TRAINING RUN")
print("=" * 80)


print("\nTraining")
print("-" * 80)

print(
    f"requested timesteps       : "
    f"{PPO_FIRST_TRAINING_TIMESTEPS}"
)

print(
    f"actual PPO timesteps      : "
    f"{ppo_model.num_timesteps}"
)

print(
    f"elapsed wall time         : "
    f"{training_elapsed_seconds:.3f} s"
)

print(
    f"effective env steps/sec   : "
    f"{ppo_model.num_timesteps / training_elapsed_seconds:.1f}"
)


print("\nPolicy action before vs after training")
print("-" * 80)

print(
    "untrained initial action :",
    ppo_probe_action,
)

print(
    "trained initial action   :",
    trained_probe_action,
)

print(
    "trained action legal     :",
    ppo_env.action_space.contains(
        trained_probe_action
    ),
)


print("\nPost-training evaluation")
print("-" * 80)

print(
    f"episodes                  : "
    f"{TRAINED_EVALUATION_EPISODES}"
)

print(
    f"successes                 : "
    f"{int(np.sum(trained_results['successes']))}"
    f"/{TRAINED_EVALUATION_EPISODES}"
)

print(
    f"success rate              : "
    f"{trained_success_rate:.3f}"
)

print(
    f"mean final distance       : "
    f"{trained_mean_final_distance:.6f} m"
)

print(
    f"mean minimum distance     : "
    f"{trained_mean_min_distance:.6f} m"
)

print(
    f"mean episode return       : "
    f"{trained_mean_return:.6f}"
)

print(
    f"mean episode length       : "
    f"{trained_mean_steps:.3f} steps"
)


print("\nPhysical comparison")
print("-" * 80)

print(
    f"random mean final distance    : "
    f"{baseline_mean_final_distance:.6f} m"
)

print(
    f"untrained mean final distance : "
    f"{untrained_mean_final_distance:.6f} m"
)

print(
    f"trained mean final distance   : "
    f"{trained_mean_final_distance:.6f} m"
)

print(
    f"distance gain vs random       : "
    f"{improvement_vs_random_distance:.6f} m"
)

print(
    f"distance gain vs untrained    : "
    f"{improvement_vs_untrained_distance:.6f} m"
)

print(
    f"relative gain vs random       : "
    f"{100.0 * relative_improvement_vs_random:.3f}%"
)

print(
    f"relative gain vs untrained    : "
    f"{100.0 * relative_improvement_vs_untrained:.3f}%"
)


print("\nReturn comparison")
print("-" * 80)

print(
    f"random mean return            : "
    f"{baseline_mean_return:.6f}"
)

print(
    f"untrained mean return         : "
    f"{untrained_mean_return:.6f}"
)

print(
    f"trained mean return           : "
    f"{trained_mean_return:.6f}"
)

print(
    f"return gain vs random         : "
    f"{return_improvement_vs_random:.6f}"
)

print(
    f"return gain vs untrained      : "
    f"{return_improvement_vs_untrained:.6f}"
)


print("\nDeterministic evaluation replay")
print("-" * 80)

print(
    "trajectory exact across runs :",
    trained_trajectory_exact,
)

print(
    "actions exact across runs    :",
    trained_actions_exact,
)


print("\nLearning evidence")
print("-" * 80)

print(
    "distance better than random   :",
    distance_better_than_random,
)

print(
    "distance better than untrained:",
    distance_better_than_untrained,
)

print(
    "success better than random    :",
    success_better_than_random,
)

print(
    "success better than untrained :",
    success_better_than_untrained,
)

print(
    "first-run improvement detected:",
    first_training_improved,
)


print("\n" + "=" * 80)

if first_training_improved:
    print(
        "STEP 8 FIRST PPO TRAINING RUN - IMPROVEMENT DETECTED"
    )
else:
    print(
        "STEP 8 FIRST PPO TRAINING RUN - "
        "NO CLEAR IMPROVEMENT YET"
    )

print("=" * 80)

print(
    "The first PPO optimization run completed through the real "
    "HydroLab-3D production Simulator without changing simulator core."
)

print(
    "The measured post-training metrics above determine whether "
    "this modest training budget is already sufficient or whether "
    "the next step should diagnose and extend the smoke test."
)

HYDROLAB-3D NOTEBOOK 11 - STEP 8 FIRST PPO TRAINING RUN

Training
--------------------------------------------------------------------------------
requested timesteps       : 20000
actual PPO timesteps      : 20224
elapsed wall time         : 20.933 s
effective env steps/sec   : 966.1

Policy action before vs after training
--------------------------------------------------------------------------------
untrained initial action : [ 0.00217546 -0.00824257 -0.0067293 ]
trained initial action   : [1.         0.29739356 1.        ]
trained action legal     : True

Post-training evaluation
--------------------------------------------------------------------------------
episodes                  : 5
successes                 : 0/5
success rate              : 0.000
mean final distance       : 1.853785 m
mean minimum distance     : 1.853785 m
mean episode return       : 4.460243
mean episode length       : 750.000 steps

Physical comparison
-----------------------------------------------------

## Step 9 - Persist the Learning-Validation Evidence

The first PPO training run has already satisfied the minimum learning-smoke-test requirement.

The trained policy reduced mean final goal distance from:

- random baseline: `4.856713 m`;
- untrained PPO: `4.861101 m`;

to:

- trained PPO: `1.853785 m`.

This corresponds to approximately:

- `61.83%` reduction relative to the random baseline;
- `61.87%` reduction relative to the untrained PPO policy.

The trained policy did not enter the `0.50 m` success sphere within the 15-second episode horizon, so the result must not be described as successful waypoint capture.

However, Notebook 11 only requires measurable learning rather than optimal task completion, and the physical improvement is already substantial and independently measurable.

We therefore do not perform an unnecessary hyperparameter sweep or arbitrarily increase the training budget.

This step preserves the compact evidence required to support the final Notebook 11 conclusion.

All files will remain strictly inside:

`results/notebooks/11_learning_interface_validation/`

### Persistent Evidence

The notebook will create:

1. `data/evaluation_metrics.csv`

   A compact comparison of:

   - random policy;
   - untrained PPO;
   - trained PPO.

   Metrics include:

   - evaluation episodes;
   - success rate;
   - mean final distance;
   - mean minimum distance;
   - mean episode return;
   - mean episode length.

2. `data/trajectory_comparison.csv`

   A trajectory record containing:

   - representative random-policy XYZ trajectory;
   - deterministic untrained-PPO XYZ trajectory;
   - deterministic trained-PPO XYZ trajectory.

   Because trajectories may have different lengths, missing samples will be represented naturally in the table rather than fabricated.

3. `plots/learning_interface_validation.png`

   A compact technical figure showing:

   - the three representative 3D trajectories;
   - final-distance comparison;
   - episode-return comparison.

4. `models/ppo_smoke_test.zip`

   The trained Stable-Baselines3 PPO model.

The saved model is retained because it provides direct reproducibility value for the demonstrated smoke test.

### Interpretation Constraint

The persistent evidence must preserve the actual result:

- random policy: no success;
- untrained PPO: no success;
- trained PPO: no success;
- trained PPO: substantially smaller goal distance and higher return.

The figure and CSV files must not imply waypoint success where none occurred.

### Validation Criteria

This step passes if:

1. every generated artifact is inside the Notebook 11 result directory;
2. the evaluation CSV contains all three policy conditions;
3. persisted values reproduce the in-memory metrics;
4. the trajectory CSV contains representative trajectories from all three policy conditions;
5. the technical figure is written successfully and is non-empty;
6. the trained PPO model is saved successfully;
7. all CSV files can be read back;
8. all persisted numerical metrics are finite;
9. no previous notebook result directory is modified;
10. no production HydroLab-3D source file is modified.

After these artifacts are validated, the remaining critical gate is the complete production regression suite.

In [12]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# 1. Notebook 11 artifact namespace
# ---------------------------------------------------------------------

NOTEBOOK11_OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "11_learning_interface_validation"
)

NOTEBOOK11_DATA_DIR = (
    NOTEBOOK11_OUTPUT_DIR
    / "data"
)

NOTEBOOK11_PLOTS_DIR = (
    NOTEBOOK11_OUTPUT_DIR
    / "plots"
)

NOTEBOOK11_MODELS_DIR = (
    NOTEBOOK11_OUTPUT_DIR
    / "models"
)


for directory in (
    NOTEBOOK11_DATA_DIR,
    NOTEBOOK11_PLOTS_DIR,
    NOTEBOOK11_MODELS_DIR,
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ---------------------------------------------------------------------
# 2. Build compact evaluation-metrics table
# ---------------------------------------------------------------------

evaluation_metrics_df = pd.DataFrame(
    [
        {
            "policy": "random",
            "evaluation_episodes": (
                BASELINE_EPISODES
            ),
            "training_timesteps": 0,
            "success_rate": (
                baseline_success_rate
            ),
            "mean_final_distance_m": (
                baseline_mean_final_distance
            ),
            "mean_min_distance_m": (
                baseline_mean_min_distance
            ),
            "mean_episode_return": (
                baseline_mean_return
            ),
            "mean_episode_steps": (
                baseline_mean_steps
            ),
        },
        {
            "policy": "untrained_ppo",
            "evaluation_episodes": (
                UNTRAINED_EVALUATION_EPISODES
            ),
            "training_timesteps": 0,
            "success_rate": (
                untrained_success_rate
            ),
            "mean_final_distance_m": (
                untrained_mean_final_distance
            ),
            "mean_min_distance_m": (
                untrained_mean_min_distance
            ),
            "mean_episode_return": (
                untrained_mean_return
            ),
            "mean_episode_steps": (
                untrained_mean_steps
            ),
        },
        {
            "policy": "trained_ppo",
            "evaluation_episodes": (
                TRAINED_EVALUATION_EPISODES
            ),
            "training_timesteps": int(
                ppo_model.num_timesteps
            ),
            "success_rate": (
                trained_success_rate
            ),
            "mean_final_distance_m": (
                trained_mean_final_distance
            ),
            "mean_min_distance_m": (
                trained_mean_min_distance
            ),
            "mean_episode_return": (
                trained_mean_return
            ),
            "mean_episode_steps": (
                trained_mean_steps
            ),
        },
    ]
)


# ---------------------------------------------------------------------
# 3. Validate evaluation table before writing
# ---------------------------------------------------------------------

assert list(
    evaluation_metrics_df[
        "policy"
    ]
) == [
    "random",
    "untrained_ppo",
    "trained_ppo",
]

numeric_metric_columns = [
    "evaluation_episodes",
    "training_timesteps",
    "success_rate",
    "mean_final_distance_m",
    "mean_min_distance_m",
    "mean_episode_return",
    "mean_episode_steps",
]

assert np.all(
    np.isfinite(
        evaluation_metrics_df[
            numeric_metric_columns
        ].to_numpy(
            dtype=float
        )
    )
)


# ---------------------------------------------------------------------
# 4. Save evaluation metrics
# ---------------------------------------------------------------------

evaluation_metrics_path = (
    NOTEBOOK11_DATA_DIR
    / "evaluation_metrics.csv"
)

evaluation_metrics_df.to_csv(
    evaluation_metrics_path,
    index=False,
)


# ---------------------------------------------------------------------
# 5. Prepare representative trajectories
# ---------------------------------------------------------------------

random_trajectory = np.asarray(
    baseline_representative_trajectory,
    dtype=float,
)

untrained_trajectory = np.asarray(
    untrained_representative_trajectory,
    dtype=float,
)

trained_trajectory = np.asarray(
    trained_representative_trajectory,
    dtype=float,
)


for name, trajectory in (
    (
        "random",
        random_trajectory,
    ),
    (
        "untrained",
        untrained_trajectory,
    ),
    (
        "trained",
        trained_trajectory,
    ),
):
    assert (
        trajectory.ndim
        == 2
    )

    assert (
        trajectory.shape[1]
        == 3
    )

    assert np.all(
        np.isfinite(
            trajectory
        )
    ), (
        f"{name} trajectory contains "
        "NaN or Inf"
    )


# ---------------------------------------------------------------------
# 6. Build trajectory comparison table
#
# Trajectories may have different lengths if a future policy terminates
# early, so each policy is represented through padded Series objects.
# ---------------------------------------------------------------------

random_trajectory_df = pd.DataFrame(
    {
        "random_x_m": (
            random_trajectory[:, 0]
        ),
        "random_y_m": (
            random_trajectory[:, 1]
        ),
        "random_z_m": (
            random_trajectory[:, 2]
        ),
    }
)

untrained_trajectory_df = pd.DataFrame(
    {
        "untrained_x_m": (
            untrained_trajectory[:, 0]
        ),
        "untrained_y_m": (
            untrained_trajectory[:, 1]
        ),
        "untrained_z_m": (
            untrained_trajectory[:, 2]
        ),
    }
)

trained_trajectory_df = pd.DataFrame(
    {
        "trained_x_m": (
            trained_trajectory[:, 0]
        ),
        "trained_y_m": (
            trained_trajectory[:, 1]
        ),
        "trained_z_m": (
            trained_trajectory[:, 2]
        ),
    }
)

trajectory_comparison_df = pd.concat(
    [
        random_trajectory_df,
        untrained_trajectory_df,
        trained_trajectory_df,
    ],
    axis=1,
)


# ---------------------------------------------------------------------
# 7. Save trajectory comparison
# ---------------------------------------------------------------------

trajectory_comparison_path = (
    NOTEBOOK11_DATA_DIR
    / "trajectory_comparison.csv"
)

trajectory_comparison_df.to_csv(
    trajectory_comparison_path,
    index=False,
)


# ---------------------------------------------------------------------
# 8. Save trained PPO smoke-test model
# ---------------------------------------------------------------------

ppo_model_base_path = (
    NOTEBOOK11_MODELS_DIR
    / "ppo_smoke_test"
)

ppo_model.save(
    str(
        ppo_model_base_path
    )
)

ppo_model_path = (
    NOTEBOOK11_MODELS_DIR
    / "ppo_smoke_test.zip"
)

assert ppo_model_path.exists()

assert (
    ppo_model_path.stat().st_size
    > 0
)


# ---------------------------------------------------------------------
# 9. Create compact technical validation figure
# ---------------------------------------------------------------------

validation_figure_path = (
    NOTEBOOK11_PLOTS_DIR
    / "learning_interface_validation.png"
)

figure = plt.figure(
    figsize=(12, 12)
)


# ---------------------------------------------------------------------
# 3D trajectory comparison
# ---------------------------------------------------------------------

trajectory_axis = figure.add_subplot(
    3,
    1,
    1,
    projection="3d",
)

trajectory_axis.plot(
    random_trajectory[:, 0],
    random_trajectory[:, 1],
    random_trajectory[:, 2],
    label="Random",
)

trajectory_axis.plot(
    untrained_trajectory[:, 0],
    untrained_trajectory[:, 1],
    untrained_trajectory[:, 2],
    label="Untrained PPO",
)

trajectory_axis.plot(
    trained_trajectory[:, 0],
    trained_trajectory[:, 1],
    trained_trajectory[:, 2],
    label="Trained PPO",
)

trajectory_axis.scatter(
    contract_initial_pose[0],
    contract_initial_pose[1],
    contract_initial_pose[2],
    marker="o",
    s=50,
    label="Start",
)

trajectory_axis.scatter(
    LEARNING_GOAL_POSITION[0],
    LEARNING_GOAL_POSITION[1],
    LEARNING_GOAL_POSITION[2],
    marker="*",
    s=100,
    label="Goal",
)

trajectory_axis.set_xlabel(
    "x [m]"
)

trajectory_axis.set_ylabel(
    "y [m]"
)

trajectory_axis.set_zlabel(
    "z [m]"
)

trajectory_axis.set_title(
    "Representative Point-Navigation Trajectories"
)

trajectory_axis.legend()


# ---------------------------------------------------------------------
# Final-distance comparison
# ---------------------------------------------------------------------

distance_axis = figure.add_subplot(
    3,
    1,
    2,
)

policy_labels = [
    "Random",
    "Untrained PPO",
    "Trained PPO",
]

final_distance_values = [
    baseline_mean_final_distance,
    untrained_mean_final_distance,
    trained_mean_final_distance,
]

distance_axis.bar(
    policy_labels,
    final_distance_values,
)

distance_axis.axhline(
    LEARNING_SUCCESS_RADIUS,
    linestyle="--",
    label="Success radius",
)

distance_axis.set_ylabel(
    "Mean final goal distance [m]"
)

distance_axis.set_title(
    "Final Goal-Distance Comparison"
)

distance_axis.legend()


# ---------------------------------------------------------------------
# Episode-return comparison
# ---------------------------------------------------------------------

return_axis = figure.add_subplot(
    3,
    1,
    3,
)

return_values = [
    baseline_mean_return,
    untrained_mean_return,
    trained_mean_return,
]

return_axis.bar(
    policy_labels,
    return_values,
)

return_axis.set_ylabel(
    "Mean episode return"
)

return_axis.set_title(
    "Task-Return Comparison"
)


figure.tight_layout()

figure.savefig(
    validation_figure_path,
    dpi=160,
    bbox_inches="tight",
)

plt.close(
    figure
)


# ---------------------------------------------------------------------
# 10. Read back persistent CSV artifacts
# ---------------------------------------------------------------------

evaluation_metrics_readback = (
    pd.read_csv(
        evaluation_metrics_path
    )
)

trajectory_comparison_readback = (
    pd.read_csv(
        trajectory_comparison_path
    )
)


# ---------------------------------------------------------------------
# 11. Validate read-back content
# ---------------------------------------------------------------------

assert (
    len(
        evaluation_metrics_readback
    )
    == 3
)

assert list(
    evaluation_metrics_readback[
        "policy"
    ]
) == [
    "random",
    "untrained_ppo",
    "trained_ppo",
]

assert (
    len(
        trajectory_comparison_readback
    )
    == max(
        len(
            random_trajectory
        ),
        len(
            untrained_trajectory
        ),
        len(
            trained_trajectory
        ),
    )
)

assert validation_figure_path.exists()

assert (
    validation_figure_path.stat().st_size
    > 0
)


# ---------------------------------------------------------------------
# 12. Verify persisted primary values against notebook memory
# ---------------------------------------------------------------------

trained_readback_row = (
    evaluation_metrics_readback.loc[
        evaluation_metrics_readback[
            "policy"
        ]
        == "trained_ppo"
    ].iloc[0]
)

random_readback_row = (
    evaluation_metrics_readback.loc[
        evaluation_metrics_readback[
            "policy"
        ]
        == "random"
    ].iloc[0]
)

untrained_readback_row = (
    evaluation_metrics_readback.loc[
        evaluation_metrics_readback[
            "policy"
        ]
        == "untrained_ppo"
    ].iloc[0]
)


assert np.isclose(
    trained_readback_row[
        "mean_final_distance_m"
    ],
    trained_mean_final_distance,
    rtol=0.0,
    atol=1e-12,
)

assert np.isclose(
    random_readback_row[
        "mean_final_distance_m"
    ],
    baseline_mean_final_distance,
    rtol=0.0,
    atol=1e-12,
)

assert np.isclose(
    untrained_readback_row[
        "mean_final_distance_m"
    ],
    untrained_mean_final_distance,
    rtol=0.0,
    atol=1e-12,
)


# ---------------------------------------------------------------------
# 13. Confirm every artifact belongs to Notebook 11
# ---------------------------------------------------------------------

artifact_paths = [
    evaluation_metrics_path,
    trajectory_comparison_path,
    validation_figure_path,
    ppo_model_path,
]

for artifact_path in artifact_paths:
    artifact_path.resolve().relative_to(
        NOTEBOOK11_OUTPUT_DIR.resolve()
    )


# ---------------------------------------------------------------------
# 14. Report
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 11 - STEP 9 PERSISTENT EVIDENCE")
print("=" * 80)


print("\nEvaluation evidence")
print("-" * 80)

print(
    evaluation_metrics_df.to_string(
        index=False
    )
)


print("\nMeasured learning result")
print("-" * 80)

print(
    f"random final distance      : "
    f"{baseline_mean_final_distance:.6f} m"
)

print(
    f"untrained final distance   : "
    f"{untrained_mean_final_distance:.6f} m"
)

print(
    f"trained final distance     : "
    f"{trained_mean_final_distance:.6f} m"
)

print(
    f"improvement vs random      : "
    f"{100.0 * relative_improvement_vs_random:.3f}%"
)

print(
    f"improvement vs untrained   : "
    f"{100.0 * relative_improvement_vs_untrained:.3f}%"
)

print(
    f"trained success rate       : "
    f"{trained_success_rate:.3f}"
)

print(
    f"success radius             : "
    f"{LEARNING_SUCCESS_RADIUS:.3f} m"
)


print("\nPersistent artifacts")
print("-" * 80)

for artifact_path in artifact_paths:

    print(
        artifact_path.relative_to(
            PROJECT_ROOT
        )
    )


print("\nArtifact validation")
print("-" * 80)

print(
    "evaluation rows          :",
    len(
        evaluation_metrics_readback
    ),
)

print(
    "trajectory rows          :",
    len(
        trajectory_comparison_readback
    ),
)

print(
    "figure bytes             :",
    validation_figure_path.stat().st_size,
)

print(
    "model bytes              :",
    ppo_model_path.stat().st_size,
)


print("\n" + "=" * 80)
print("STEP 9 PERSISTENT LEARNING EVIDENCE PASSED")
print("=" * 80)

print(
    "Notebook 11 now contains compact persistent evidence for the "
    "random, untrained-PPO, and trained-PPO comparison."
)

print(
    "The trained policy did not reach the 0.50 m success region, "
    "but reduced mean final goal distance by approximately "
    f"{100.0 * relative_improvement_vs_random:.2f}% relative to random control."
)

print(
    "All generated files are isolated under "
    "results/notebooks/11_learning_interface_validation/."
)

HYDROLAB-3D NOTEBOOK 11 - STEP 9 PERSISTENT EVIDENCE

Evaluation evidence
--------------------------------------------------------------------------------
       policy  evaluation_episodes  training_timesteps  success_rate  mean_final_distance_m  mean_min_distance_m  mean_episode_return  mean_episode_steps
       random                   50                   0           0.0               4.856713             4.856631            -7.269693               750.0
untrained_ppo                    5                   0           0.0               4.861101             4.861101             0.209299               750.0
  trained_ppo                    5               20224           0.0               1.853785             1.853785             4.460243               750.0

Measured learning result
--------------------------------------------------------------------------------
random final distance      : 4.856713 m
untrained final distance   : 4.861101 m
trained final distance     : 1.853785 m
im

## Step 10 - Run the Final Production Regression Suite

Notebook 11 has now completed the learning-specific validation path:

- the production Simulator contract was verified;
- the notebook-local learning task was defined explicitly;
- the Gymnasium wrapper passed direct validation;
- Gymnasium and Stable-Baselines3 environment checks passed;
- 7,500 bounded-random stress transitions completed without numerical failure;
- a formal 50-episode random baseline was established;
- Stable-Baselines3 PPO successfully interfaced with the production simulator;
- 20,224 PPO training transitions completed;
- the trained policy reduced mean final goal distance from approximately 4.86 m to 1.85 m;
- learning-validation evidence was persisted exclusively under the Notebook 11 result namespace.

The remaining critical gate is production regression.

Notebook 10 ended with the inherited regression baseline:

- 15 dedicated Simulator tests passed;
- 167 complete repository tests passed;
- 0 failures.

Notebook 11 has deliberately kept all learning-specific code inside the notebook, so no production source change was expected.

Nevertheless, successful learning execution is not enough to conclude the notebook.

The complete HydroLab-3D test suite must still pass in the actual project environment.

### Regression Procedure

This step launches pytest through the same Python interpreter running the notebook:

`sys.executable`

and executes from the HydroLab-3D project root.

The repository `src/` directory is explicitly included in `PYTHONPATH` so the tests exercise the local production package.

Two regression commands are executed:

1. the dedicated production Simulator test module;
2. the complete repository test suite.

The expected inherited counts are:

- dedicated Simulator suite: 15 passed;
- complete repository suite: 167 passed.

No test file, source file, dependency file, result artifact, or model is modified by this step.

### Validation Criteria

This step passes only if:

1. the dedicated Simulator pytest process exits with return code 0;
2. the dedicated suite reports 15 passing tests;
3. the complete repository pytest process exits with return code 0;
4. the complete suite reports 167 passing tests;
5. neither run reports a test failure;
6. the tests execute against the current HydroLab-3D repository and active virtual environment.

If this regression gate passes, Notebook 11 will have all evidence required for its final conclusion.

In [13]:
import os
import re
import subprocess
import sys


# ---------------------------------------------------------------------
# 1. Regression configuration
# ---------------------------------------------------------------------

EXPECTED_SIMULATOR_TESTS = 15
EXPECTED_FULL_TESTS = 167

regression_environment = os.environ.copy()

existing_pythonpath = (
    regression_environment.get(
        "PYTHONPATH",
        "",
    )
)

src_path_string = str(
    SRC_DIR.resolve()
)

if existing_pythonpath:
    regression_environment[
        "PYTHONPATH"
    ] = (
        src_path_string
        + os.pathsep
        + existing_pythonpath
    )
else:
    regression_environment[
        "PYTHONPATH"
    ] = src_path_string


# ---------------------------------------------------------------------
# 2. Helper for pytest execution
# ---------------------------------------------------------------------

def run_pytest_regression(
    test_target,
):
    command = [
        sys.executable,
        "-m",
        "pytest",
        test_target,
        "-q",
    ]

    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        env=regression_environment,
        capture_output=True,
        text=True,
        check=False,
    )

    combined_output = (
        result.stdout
        + "\n"
        + result.stderr
    )

    match = re.search(
        r"(\d+)\s+passed",
        combined_output,
    )

    passed_count = (
        int(
            match.group(1)
        )
        if match
        else None
    )

    return {
        "command": command,
        "returncode": (
            result.returncode
        ),
        "stdout": (
            result.stdout
        ),
        "stderr": (
            result.stderr
        ),
        "passed_count": (
            passed_count
        ),
    }


# ---------------------------------------------------------------------
# 3. Dedicated production Simulator regression
# ---------------------------------------------------------------------

simulator_regression = (
    run_pytest_regression(
        "tests/test_simulator.py"
    )
)


# ---------------------------------------------------------------------
# 4. Complete repository regression
# ---------------------------------------------------------------------

full_regression = (
    run_pytest_regression(
        "tests"
    )
)


# ---------------------------------------------------------------------
# 5. Print raw pytest evidence before assertions
# ---------------------------------------------------------------------

print("=" * 80)
print("HYDROLAB-3D NOTEBOOK 11 - STEP 10 FINAL REGRESSION")
print("=" * 80)


print("\nRuntime")
print("-" * 80)

print(
    "Python executable :",
    sys.executable,
)

print(
    "Project root      :",
    PROJECT_ROOT,
)

print(
    "PYTHONPATH src    :",
    SRC_DIR,
)


print("\nDedicated Simulator regression")
print("-" * 80)

print(
    "command           :",
    " ".join(
        simulator_regression[
            "command"
        ]
    ),
)

print(
    "return code       :",
    simulator_regression[
        "returncode"
    ],
)

print(
    "parsed pass count :",
    simulator_regression[
        "passed_count"
    ],
)

print("\nstdout:")

if simulator_regression[
    "stdout"
].strip():
    print(
        simulator_regression[
            "stdout"
        ].strip()
    )
else:
    print(
        "<NO STDOUT>"
    )

if simulator_regression[
    "stderr"
].strip():
    print("\nstderr:")
    print(
        simulator_regression[
            "stderr"
        ].strip()
    )


print("\nComplete repository regression")
print("-" * 80)

print(
    "command           :",
    " ".join(
        full_regression[
            "command"
        ]
    ),
)

print(
    "return code       :",
    full_regression[
        "returncode"
    ],
)

print(
    "parsed pass count :",
    full_regression[
        "passed_count"
    ],
)

print("\nstdout:")

if full_regression[
    "stdout"
].strip():
    print(
        full_regression[
            "stdout"
        ].strip()
    )
else:
    print(
        "<NO STDOUT>"
    )

if full_regression[
    "stderr"
].strip():
    print("\nstderr:")
    print(
        full_regression[
            "stderr"
        ].strip()
    )


# ---------------------------------------------------------------------
# 6. Strict regression assertions
# ---------------------------------------------------------------------

assert (
    simulator_regression[
        "returncode"
    ]
    == 0
), (
    "Dedicated Simulator regression failed."
)

assert (
    simulator_regression[
        "passed_count"
    ]
    == EXPECTED_SIMULATOR_TESTS
), (
    "Dedicated Simulator regression count changed: "
    f"expected {EXPECTED_SIMULATOR_TESTS}, "
    f"observed "
    f"{simulator_regression['passed_count']}."
)


assert (
    full_regression[
        "returncode"
    ]
    == 0
), (
    "Complete HydroLab-3D regression failed."
)

assert (
    full_regression[
        "passed_count"
    ]
    == EXPECTED_FULL_TESTS
), (
    "Complete regression count changed: "
    f"expected {EXPECTED_FULL_TESTS}, "
    f"observed "
    f"{full_regression['passed_count']}."
)


# ---------------------------------------------------------------------
# 7. Explicit failure-string checks
# ---------------------------------------------------------------------

simulator_combined_output = (
    simulator_regression[
        "stdout"
    ]
    + "\n"
    + simulator_regression[
        "stderr"
    ]
).lower()

full_combined_output = (
    full_regression[
        "stdout"
    ]
    + "\n"
    + full_regression[
        "stderr"
    ]
).lower()

assert (
    " failed"
    not in simulator_combined_output
)

assert (
    " failed"
    not in full_combined_output
)


# ---------------------------------------------------------------------
# 8. Final regression summary
# ---------------------------------------------------------------------

print("\nRegression summary")
print("-" * 80)

print(
    f"Simulator tests : "
    f"{simulator_regression['passed_count']} passed"
)

print(
    f"Full test suite : "
    f"{full_regression['passed_count']} passed"
)

print(
    "Failures        : 0"
)


print("\n" + "=" * 80)
print("STEP 10 FINAL PRODUCTION REGRESSION PASSED")
print("=" * 80)

print(
    "Notebook 11 preserved the inherited HydroLab-3D production "
    "regression baseline."
)

print(
    "All 15 dedicated Simulator tests and all 167 repository tests "
    "pass in the current project environment."
)

print(
    "The learning-interface validation introduced no detected "
    "regression into the production simulator stack."
)

HYDROLAB-3D NOTEBOOK 11 - STEP 10 FINAL REGRESSION

Runtime
--------------------------------------------------------------------------------
Python executable : /home/agniacolyte/HydroLab-3D/.venv/bin/python
Project root      : /home/agniacolyte/HydroLab-3D
PYTHONPATH src    : /home/agniacolyte/HydroLab-3D/src

Dedicated Simulator regression
--------------------------------------------------------------------------------
command           : /home/agniacolyte/HydroLab-3D/.venv/bin/python -m pytest tests/test_simulator.py -q
return code       : 0
parsed pass count : 15

stdout:
...............                                                          [100%]
15 passed in 0.48s

Complete repository regression
--------------------------------------------------------------------------------
command           : /home/agniacolyte/HydroLab-3D/.venv/bin/python -m pytest tests -q
return code       : 0
parsed pass count : 167

stdout:
................................................................

# Conclusion - Learning Interface Validation

Notebook 11 successfully validated that the finished HydroLab-3D production simulator can support a standard reinforcement-learning workflow **without redesigning the simulator core**.

The objective of this notebook was deliberately narrow.

It was not intended to establish a state-of-the-art UUV reinforcement-learning controller, solve general underwater navigation, or demonstrate sensor-based autonomous localization.

Instead, the notebook asked a systems-level integration question:

> Can an external learning algorithm interact with the existing HydroLab-3D `Simulator`, using its established state and generalized-wrench interfaces, while keeping learning-specific concepts outside the production core?

The evidence collected in this notebook supports a clear **yes**.

---

## 1. Production Simulator Interface

The learning validation used the actual production:

`hydrolab3d.core.Simulator`

with the production constructor:

`Simulator(initial_pose, initial_body_velocity, dt, dynamics_kwargs, sensor_config, sensor_seed)`

The validated runtime contract is:

`reset()`

returns production ground truth containing:

- simulation time;
- six-element pose;
- six-element body velocity.

`step(action)`

accepts a finite six-element generalized body-frame wrench:

$$
\tau =
[F_x,F_y,F_z,M_x,M_y,M_z]^T
$$

and returns separate:

- `truth`;
- `observation`.

The sensor branch contains:

- depth;
- heading;
- DVL;
- gyroscope;
- accelerometer.

This separation between simulator truth and generated sensor measurements remained intact throughout Notebook 11.

No learning-specific concept was inserted into this production interface.

---

## 2. Notebook-Local Learning Task

A deliberately minimal fixed-waypoint task was constructed around the production simulator.

The initial UUV position was:

$$
\mathbf{p}_0 =
[0,0,-5]^T \text{ m}
$$

and the fixed target was:

$$
\mathbf{p}_{goal} =
[6,-2,-4]^T \text{ m}.
$$

The initial Euclidean goal distance was:

$$
d_0 = 6.403124237 \text{ m}.
$$

The learning policy received a 12-element truth-derived observation:

$$
o_t =
[
\Delta x,
\Delta y,
\Delta z,
\phi,
\theta,
\psi,
u,
v,
w,
p,
q,
r
]^T.
$$

The first three components represented world-frame displacement from the vehicle to the goal.

The remaining components came directly from the production truth state:

- roll, pitch, yaw;
- body-frame linear velocity;
- body-frame angular velocity.

This observation was intentionally based on simulator truth.

Accordingly, Notebook 11 demonstrates a **learning-interface and control smoke test**, not a complete sensor-based navigation system.

No claim is made that HydroLab-3D currently provides a validated state estimator or localization pipeline for this task.

---

## 3. Learning Action to Production Action

The learning policy operated in a normalized continuous action space:

$$
a_t =
[a_x,a_y,a_z]^T,
\qquad
a_i \in [-1,1].
$$

Notebook-local mapping converted this action into the existing production wrench:

$$
\tau_t =
[
20a_x,
20a_y,
20a_z,
0,
0,
0
]^T.
$$

Thus the learner controlled translational body-frame forces with limits:

$$
|F_x|,\ |F_y|,\ |F_z|
\le 20\text{ N}.
$$

The production six-axis action contract was therefore preserved exactly.

The learning layer did not require a new simulator action representation.

The policy simply translated its reduced action into the already-supported generalized wrench.

---

## 4. Reward and Episode Contract

The notebook-local reward was:

$$
r_t =
5(d_{t-1}-d_t)
-
0.01\|a_t\|_2^2
-
0.01
+
r_{success}.
$$

The reward therefore favored:

- progress toward the waypoint;
- lower control effort;
- shorter completion time.

Entering the success region added a bonus of:

$$
+25.
$$

Waypoint success was defined by:

$$
d_t \le 0.50\text{ m}.
$$

The maximum episode duration was:

`750 steps`

with production timestep:

$$
\Delta t = 0.02\text{ s}.
$$

Therefore the maximum simulated episode duration was:

$$
750(0.02)=15\text{ s}.
$$

Gymnasium semantics were kept explicit:

- waypoint arrival → `terminated = True`;
- horizon exhaustion → `truncated = True`.

No artificial physical world boundary was introduced.

---

## 5. Gymnasium Compatibility

A temporary notebook-local `HydroLabLearningEnv` was built around the production simulator.

The wrapper exposed:

- action space: `Box(-1.0, 1.0, (3,), float32)`;
- observation space: `Box(-inf, inf, (12,), float32)`.

The environment successfully passed:

- Gymnasium's environment checker;
- Stable-Baselines3's environment checker.

Gymnasium emitted warnings about infinite observation-space limits.

These were warnings rather than contract failures.

The observation space was intentionally left physically unbounded because Notebook 11 did not introduce artificial production-world limits merely to satisfy a tighter learning-space definition.

Runtime finiteness was instead checked explicitly throughout the experiment.

---

## 6. Random-Policy Stress Validation

Before training, the learning interface was exercised through:

`10 complete random-policy episodes`

for a total of:

`7,500 production Simulator transitions`.

All 10 episodes reached the 750-step horizon.

Results included:

- successful episodes: `0 / 10`;
- truncated episodes: `10 / 10`;
- mean final distance: `4.918944 m`;
- mean minimum distance: `4.918944 m`;
- mean return: `-7.615735`.

Across the complete stress test:

- observations remained finite;
- rewards remained finite;
- production wrenches remained finite;
- translational forces remained inside their declared limits;
- commanded rotational moments remained zero;
- simulator time remained consistent with the production timestep;
- repeated reset restored the exact benchmark initial state.

Malformed actions were also tested explicitly.

The wrapper correctly rejected:

- wrong-shaped actions;
- actions containing `NaN`;
- actions containing `Inf`.

Finite actions outside the normalized policy range were safely clipped before wrench conversion.

For example:

`[2.0, -3.0, 0.5]`

was converted into:

`[20, -20, 10, 0, 0, 0] N`.

This confirmed that the temporary learning boundary did not allow the policy to exceed its declared translational force limits.

---

## 7. Formal Random Baseline

A separate 50-episode random-policy evaluation established the formal pre-training baseline.

The random policy achieved:

- successes: `0 / 50`;
- success rate: `0.000`;
- mean final distance: `4.856713 m`;
- median final distance: `4.859576 m`;
- mean minimum distance: `4.856631 m`;
- mean episode return: `-7.269693`;
- median episode return: `-7.331489`;
- mean episode length: `750 steps`.

The best random final distance observed across the 50 episodes was:

`4.039199 m`.

Thus bounded random control consistently failed to approach the `0.50 m` waypoint-success region.

---

## 8. Untrained PPO Reference

Stable-Baselines3 PPO was then connected directly to the notebook-local environment.

The smoke-test policy used:

- `MlpPolicy`;
- hidden architecture `[64, 64]`;
- CPU execution;
- learning rate `3e-4`;
- `n_steps = 256`;
- batch size `64`;
- `10` PPO epochs;
- `gamma = 0.99`;
- `gae_lambda = 0.95`;
- `clip_range = 0.2`;
- explicit learning seed `22017`.

Before any gradient update, the initialized policy produced a legal initial action:

`[0.00217546, -0.00824257, -0.00672930]`.

Five deterministic untrained rollouts produced identical trajectories and actions, as expected for the fixed deterministic benchmark.

The untrained PPO reference achieved:

- successes: `0 / 5`;
- success rate: `0.000`;
- mean final distance: `4.861101 m`;
- mean minimum distance: `4.861101 m`;
- mean episode return: `0.209299`;
- mean episode length: `750 steps`.

Its physical endpoint performance was therefore almost identical to the random baseline.

This gave Notebook 11 a clean pre-training neural-policy reference.

---

## 9. PPO Learning Smoke Test

PPO was trained for a requested:

`20,000 timesteps`.

Because Stable-Baselines3 collects complete rollout buffers, the actual training count was:

`20,224 timesteps`.

Training completed in approximately:

`20.933 s`

on CPU, corresponding to approximately:

`966 environment steps/s`.

No numerical instability or simulator failure occurred during optimization.

After training, the deterministic initial policy action became:

`[1.00000000, 0.29739356, 1.00000000]`.

The learned policy therefore changed substantially from its initialized near-zero action behavior.

Five deterministic post-training evaluations again produced identical trajectories.

The trained policy achieved:

- successes: `0 / 5`;
- success rate: `0.000`;
- mean final distance: `1.853785 m`;
- mean minimum distance: `1.853785 m`;
- mean episode return: `4.460243`;
- mean episode length: `750 steps`.

The policy did **not** reach the declared:

`0.50 m`

success radius.

Therefore this notebook does not claim successful waypoint capture.

However, the physical improvement was substantial.

Relative to the random-policy baseline:

$$
4.856713 - 1.853785
=
3.002928\text{ m}
$$

of mean final-distance improvement was observed.

That corresponds to approximately:

$$
61.83\%
$$

reduction in mean final distance.

Relative to the untrained PPO policy:

$$
4.861101 - 1.853785
=
3.007316\text{ m},
$$

corresponding to approximately:

$$
61.87\%
$$

improvement.

Mean return also increased from:

- random: `-7.269693`;
- untrained PPO: `0.209299`;

to:

- trained PPO: `4.460243`.

The smoke test therefore produced measurable learning in both physical task progress and the declared task objective.

---

## 10. Persistent Notebook 11 Evidence

Material evidence was stored only under:

`results/notebooks/11_learning_interface_validation/`

The generated artifacts are:

`data/evaluation_metrics.csv`

containing the quantitative random, untrained-PPO, and trained-PPO comparison.

`data/trajectory_comparison.csv`

containing representative XYZ trajectories for all three policy conditions.

`plots/learning_interface_validation.png`

containing the trajectory, final-distance, and episode-return comparison.

`models/ppo_smoke_test.zip`

containing the trained Stable-Baselines3 smoke-test model.

The persisted evaluation table records:

| Policy | Training Timesteps | Success Rate | Mean Final Distance | Mean Minimum Distance | Mean Return |
|---|---:|---:|---:|---:|---:|
| Random | 0 | 0.000 | 4.856713 m | 4.856631 m | -7.269693 |
| Untrained PPO | 0 | 0.000 | 4.861101 m | 4.861101 m | 0.209299 |
| Trained PPO | 20,224 | 0.000 | 1.853785 m | 1.853785 m | 4.460243 |

The artifacts preserve the actual limitation that waypoint success remained zero.

No artifact represents the trained policy as having completed the waypoint task.

---

## 11. Production Regression

The final requirement was to verify that Notebook 11 had not damaged the existing HydroLab-3D production stack.

The dedicated Simulator regression suite produced:

`15 passed in 0.48 s`

with:

`0 failures`.

The complete repository regression suite produced:

`167 passed in 1.08 s`

with:

`0 failures`.

This exactly preserved the inherited Notebook 10 regression baseline:

- `15 / 15` Simulator tests passing;
- `167 / 167` complete repository tests passing.

No production regression was detected.

---

# Final Engineering Conclusion

Notebook 11 validates that **HydroLab-3D already possesses a usable learning-facing systems boundary**.

The production simulator did not require PPO-specific methods, Gymnasium dependencies, reward logic, episode management, policy observations, neural-network code, or learning-framework abstractions.

Instead, the integration remained cleanly layered:

    Stable-Baselines3 PPO
            ↓
    notebook-local Gymnasium wrapper
            ↓
    normalized task action
            ↓
    notebook-local wrench mapping
            ↓
    production 6D generalized wrench
            ↓
    HydroLab-3D Simulator
            ↓
    production dynamics + sensors
            ↓
    production truth
            ↓
    notebook-local task observation/reward

This separation is the central architectural result.

HydroLab-3D remains responsible for:

- deterministic simulation time;
- six-degree-of-freedom state evolution;
- hydrodynamic and hydrostatic physics;
- environmental effects;
- generalized-wrench actuation;
- truth-state ownership;
- sensor generation.

The learning layer remains responsible for:

- policy observation construction;
- reduced action-space definition;
- reward definition;
- task success criteria;
- episode termination;
- Gymnasium compatibility;
- PPO optimization;
- task-level evaluation.

That boundary allowed a standard external RL library to execute more than twenty thousand learning transitions through the real production simulator without core redesign and without breaking any existing production test.

The resulting PPO policy did not solve the waypoint task completely within the deliberately modest smoke-test budget.

It did, however, reduce mean final goal distance from approximately:

`4.86 m`

to:

`1.85 m`,

a reduction of approximately:

`61.8%`

relative to both principal pre-training references.

That is sufficient for the purpose of this notebook: it demonstrates that optimization can propagate successfully through the production HydroLab-3D simulation loop and produce measurable behavioral improvement.

Accordingly, the validated conclusion of Notebook 11 is:

> **HydroLab-3D can support external learning algorithms through a thin task-specific adapter while keeping the production simulator independent of Gymnasium, Stable-Baselines3, PPO, Torch, reward definitions, and task semantics. No core redesign was required, measurable PPO learning occurred, and the full inherited production regression suite remained at 167/167 passing tests.**

Future work may build richer learning tasks on this interface, including randomized initial conditions, sensor-derived policy observations, localization or state-estimation pipelines, rotational control, disturbance randomization, multi-goal tasks, curriculum learning, domain randomization, and multi-agent learning.

Those are extensions of the validated interface rather than prerequisites for it.

---